In [ ]:
!pip install -q optuna pytorch-tabnet catboost shap pingouin statsmodels
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
PROJECT_ROOT = Path('/content/drive/MyDrive/SBS_data')

In [ ]:
import pandas as pd
import numpy as np
import os, json, warnings
warnings.filterwarnings('ignore')

BASE = str(PROJECT_ROOT)
SUBDIRS = [
    '0_Data/raw', '0_Data/processed', '0_Data/lookups', '0_Data/scalers',
    '1_Level1_CV/results', '1_Level1_CV/analysis', '1_Level1_CV/figures', '1_Level1_CV/tables',
    '2_Level2_Holdout/results', '2_Level2_Holdout/figures', '2_Level2_Holdout/tables',
    '3_Level3_External/results', '3_Level3_External/figures', '3_Level3_External/tables',
    '4_Models', '5_Figures', '6_Manuscript', '7_HuggingFace/models'
]
for sub in SUBDIRS:
    os.makedirs(os.path.join(BASE, sub), exist_ok=True)
print(f"✓ Folder structure created under: {BASE}")

RAW_CSV = 'dataset_raw_pure_polymers.csv'  # adjust if needed
df = pd.read_csv(RAW_CSV)
print(f"✓ Loaded: {df.shape[0]} rows × {df.shape[1]} columns")

df.to_csv(os.path.join(BASE, '0_Data/raw/dataset_raw_pure_polymers.csv'), index=False)
print(f"✓ Raw CSV saved to Drive")

print("\n" + "="*60)
print("VERIFICATION")
print("="*60)

n_papers = df['Paper_ID'].nunique()
n_polymers = df['Polymer_type'].nunique()
print(f"Rows: {df.shape[0]}  |  Papers: {n_papers}  |  Polymers: {n_polymers}")

blend_mask = df['Blend_ratio'].apply(lambda x: str(x) not in ['NR', 'nan', '1', '1.0'])
n_blends = blend_mask.sum()
print(f"Blend rows (should be 0): {n_blends}")
if n_blends > 0:
    print("  ⚠ BLEND ROWS FOUND:")
    print(df.loc[blend_mask, ['Paper_ID','Polymer_type','Blend_ratio']].to_string())

checks = {
    'Air_pressure_bar_std': (0.10, 6.20),
    'Feed_rate_mLh_std':    (0.10, 80.0),
    'Working_distance_cm':  (5.0,  60.0),
    'Polymer_conc_wt':      (0.5,  35.0),
    'Fiber_diameter_nm_std': (30,  5000),
}
for col, (lo, hi) in checks.items():
    vals = pd.to_numeric(df[col], errors='coerce')
    cmin, cmax = vals.min(), vals.max()
    flag = "✓" if cmin >= lo * 0.9 and cmax <= hi * 1.1 else "⚠"
    print(f"  {flag} {col}: [{cmin:.2f}, {cmax:.2f}]  (expected [{lo}, {hi}])")

critical = ['Polymer_conc_wt', 'Polymer_MW_kDa', 'Solvent_system',
            'Air_pressure_bar_std', 'Feed_rate_mLh_std',
            'Working_distance_cm', 'Fiber_diameter_nm_std']
for col in critical:
    nr_count = (df[col].astype(str).str.strip().str.upper() == 'NR').sum()
    nan_count = df[col].isna().sum()
    flag = "✓" if (nr_count + nan_count) == 0 else "⚠"
    print(f"  {flag} {col}: NR={nr_count}, NaN={nan_count}")

EXCLUDED = ['Barros2024b','Cao2022','Carlos2022','Farias2021','Kasiri2020',
            'Paclijan2021','Scaffaro2023','Zhang2025c','Paschoalin2017',
            'Huang2024','Wojasinski2014','Rempel2019','Zhang2025b',
            'Liu2024','Dias2022']
found = df[df['Paper_ID'].isin(EXCLUDED)]['Paper_ID'].unique()
print(f"  {'✓' if len(found)==0 else '⚠'} Excluded papers present: {list(found) if len(found) else 'None'}")

print("\n--- Gelatin Audit (3 distinct materials — NEVER merge) ---")
for gtype in ['Gelatin', 'gelatin', 'Gelatin (FSG)']:
    subset = df[df['Polymer_type'] == gtype]
    if len(subset) > 0:
        mws = subset['Polymer_MW_kDa'].unique()
        papers = subset['Paper_ID'].unique()
        print(f"  '{gtype}': {len(subset)} rows, MW={mws}, papers={list(papers)}")

print("\n--- Polymer × MW combinations ---")
poly_mw = df.groupby('Polymer_type').agg(
    n_rows=('Paper_ID','size'),
    n_papers=('Paper_ID','nunique'),
    MW_values=('Polymer_MW_kDa', lambda x: sorted(x.unique().tolist()))
).sort_values('n_rows', ascending=False)
print(poly_mw.to_string())

print("\n--- Solvent systems (raw, pre-normalization) ---")
print(df['Solvent_system'].value_counts().to_string())

print("\n" + "="*60)
print("Cell 0 complete. Review output, then run Cell 1.")
print("="*60)

In [ ]:
print("=== POLYMER RENAME ===")
before = df['Polymer_type'].value_counts().get('gelatin', 0)
df.loc[df['Polymer_type'] == 'gelatin', 'Polymer_type'] = 'Gelatin (HMWFG)'
after = df['Polymer_type'].value_counts().get('Gelatin (HMWFG)', 0)
print(f"  'gelatin' → 'Gelatin (HMWFG)': {before} → {after} rows")

for g in ['Gelatin', 'Gelatin (FSG)', 'Gelatin (HMWFG)']:
    sub = df[df['Polymer_type'] == g]
    print(f"  '{g}': {len(sub)} rows, MW={sub['Polymer_MW_kDa'].unique()}")
print(f"  Total polymers now: {df['Polymer_type'].nunique()}")

print("\n=== SOLVENT × RATIO INSPECTION (pre-normalization) ===")
solv_ratio = df.groupby(['Solvent_system', 'Solvent_ratio']).agg(
    n=('Paper_ID', 'size'),
    papers=('Paper_ID', lambda x: list(x.unique()))
).reset_index()
print(solv_ratio.to_string(index=False))

print("\n=== SOLVENT NORMALIZATION ===")

SOLVENT_MAP = {
    'acetic acid':       'acetic_acid',
    'formic acid':       'formic_acid',

    'acetic_acid_water':           'acetic_acid/water',
    'acetic acid/water':           'acetic_acid/water',
    'acetic acid/water (20% v/v)': 'acetic_acid/water',
    'acetic_acid/water':           'acetic_acid/water',  # already correct

    'acetone/DMF':      'DMF/acetone',     # DMF before acetone
    'acetone/HAc':      'acetone/acetic_acid',

    'acetic acid/formic acid': 'acetic_acid/formic_acid',
}

df['Solvent_system_raw'] = df['Solvent_system']  # keep original
df['Solvent_system'] = df['Solvent_system'].replace(SOLVENT_MAP)

changed = df[df['Solvent_system'] != df['Solvent_system_raw']]
print(f"  Rows changed: {len(changed)}")
change_summary = changed.groupby(['Solvent_system_raw', 'Solvent_system']).size()
for (old, new), n in change_summary.items():
    print(f"    '{old}' → '{new}': {n} rows")

print(f"\n=== SOLVENT INVENTORY (post-normalization) ===")
print(f"  Unique solvents: {df['Solvent_system'].nunique()} (was 33)")
print(df['Solvent_system'].value_counts().to_string())

print("\n=== ACETIC ACID/WATER RATIO AUDIT ===")
aaw = df[df['Solvent_system'] == 'acetic_acid/water']
print(aaw.groupby(['Solvent_ratio', 'Paper_ID']).size().reset_index(name='n').to_string(index=False))

print("\n=== DMF/ACETONE RATIO AUDIT ===")
dma = df[df['Solvent_system'] == 'DMF/acetone']
print(dma.groupby(['Solvent_ratio', 'Paper_ID']).size().reset_index(name='n').to_string(index=False))

df.to_csv(os.path.join(BASE, '0_Data/processed/step1_names.csv'), index=False)
print(f"\n✓ Checkpoint saved: step1_names.csv")
print("="*60)
print("Cell 1 complete. Review solvent ratios, then run Cell 2.")
print("="*60)

In [ ]:
mask_kramar = (df['Paper_ID'] == 'KramarCA_preprint') & (df['Solvent_system'] == 'DMF/acetone')
print("=== RATIO FIX: KramarCA_preprint ===")
print(f"  Rows affected: {mask_kramar.sum()}")
print(f"  Before: {df.loc[mask_kramar, 'Solvent_ratio'].values}")
df.loc[mask_kramar, 'Solvent_ratio'] = '10:90'  # was 90:10 as acetone:DMF, now DMF:acetone
print(f"  After:  {df.loc[mask_kramar, 'Solvent_ratio'].values}")

MH_PARAMS = {
    'PLA':           (5.45e-4,    0.73,  'chloroform_25C',   'Dorgan2005'),
    'PLLA':          (5.45e-4,    0.73,  'chloroform_25C',   'Dorgan2005'),
    'PDLLA':         (2.21e-4,    0.77,  'chloroform_25C',   'Polymer_Handbook'),
    'PBS':           (3.98e-4,    0.74,  'chloroform_30C',   'Polymer_Handbook'),
    'PCL':           (1.395e-4,   0.786, 'chloroform_25C',   'Polymer_Handbook'),
    'PMMA':          (4.85e-5,    0.80,  'chloroform_25C',   'Polymer_Handbook'),
    'PS':            (7.10e-5,    0.74,  'chloroform_25C',   'Polymer_Handbook'),
    'PEO':           (1.25e-4,    0.78,  'water_25C',        'Polymer_Handbook'),
    'PVP':           (1.40e-4,    0.70,  'water_25C',        'Polymer_Handbook'),
    'PVA':           (6.51e-4,    0.628, 'water_25C',        'Polymer_Handbook'),
    'PVDF':          (2.00e-4,    0.675, 'DMF_25C',          'Polymer_Handbook'),
    'PAN':           (2.43e-4,    0.75,  'DMF_25C',          'Polymer_Handbook'),
    'PSf':           (2.27e-4,    0.574, 'DMF_25C',          'Kim1987'),
    'PA6':           (2.26e-4,    0.82,  'formic_acid_25C',  'Polymer_Handbook'),
    'PA66':          (3.53e-4,    0.786, 'formic_acid_25C',  'Polymer_Handbook'),
    'PC':            (3.05e-4,    0.76,  'chloroform_25C',   'Polymer_Handbook'),
    'CA':            (1.37e-4,    0.72,  'acetone_25C',      'Polymer_Handbook'),
    'PHBV':          (1.18e-4,    0.79,  'chloroform_30C',   'Polymer_Handbook'),
    'PBAT':          (2.44e-4,    0.69,  'chloroform_25C',   'Weng2013'),
    'PVB':           (2.00e-4,    0.65,  'ethanol_25C',      'approx_PVB_lit'),
    'PU':            (1.17e-4,    0.72,  'DMF_25C',          'Polymer_Handbook'),
    'TPU':           (1.17e-4,    0.72,  'DMF_25C',          'same_as_PU'),
    'THV':           (2.00e-4,    0.65,  'acetone_25C',      'approx_fluoropolymer'),
    'Gelatin':       (1.66e-5,    0.885, 'water_35C',        'Pouradier1950'),
    'Gelatin (FSG)': (1.66e-5,    0.885, 'water_35C',        'Pouradier1950'),
    'Gelatin (HMWFG)':(1.66e-5,   0.885, 'water_35C',        'Pouradier1950'),
    'Zein':          (5.00e-5,    0.667, 'ethanol_25C',      'Selling2003_approx'),
    'RSF':           (1.05e-4,    0.70,  'water_25C',        'approx_silk_fibroin'),
    'UHMWPE':        (6.20e-5,    0.70,  'decalin_135C',     'Polymer_Handbook'),
}

missing_mh = set(df['Polymer_type'].unique()) - set(MH_PARAMS.keys())
print(f"\n=== MARK-HOUWINK LOOKUP ===")
print(f"  Polymers in data: {df['Polymer_type'].nunique()}")
print(f"  Polymers in MH table: {len(MH_PARAMS)}")
print(f"  Missing: {missing_mh if missing_mh else 'None'}")

def compute_eta(row):
    poly = row['Polymer_type']
    mw_kda = row['Polymer_MW_kDa']
    K, a, _, _ = MH_PARAMS[poly]
    M_da = mw_kda * 1000  # kDa → Da
    eta = K * (M_da ** a)  # dL/g
    return eta

df['intrinsic_viscosity_dLg'] = df.apply(compute_eta, axis=1)

print(f"\n=== INTRINSIC VISCOSITY [η] (dL/g) per Polymer × MW ===")
eta_summary = df.groupby(['Polymer_type', 'Polymer_MW_kDa']).agg(
    eta=('intrinsic_viscosity_dLg', 'first'),
    n_rows=('Paper_ID', 'size')
).reset_index()
eta_summary = eta_summary.sort_values('eta', ascending=False)
print(eta_summary.to_string(index=False))

eta_min = df['intrinsic_viscosity_dLg'].min()
eta_max = df['intrinsic_viscosity_dLg'].max()
print(f"\n  [η] range: {eta_min:.4f} – {eta_max:.4f} dL/g")
outliers = df[(df['intrinsic_viscosity_dLg'] < 0.01) | (df['intrinsic_viscosity_dLg'] > 20)]
if len(outliers) > 0:
    print(f"  ⚠ {len(outliers)} rows outside [0.01, 20] range:")
    print(outliers[['Paper_ID','Polymer_type','Polymer_MW_kDa','intrinsic_viscosity_dLg']].to_string())
else:
    print(f"  ✓ All values within expected range")

rho_approx = 1.0
df['c_star_wt_pct'] = 10.0 / (df['intrinsic_viscosity_dLg'] * rho_approx)
df['c_over_cstar'] = df['Polymer_conc_wt'] / df['c_star_wt_pct']

print(f"\n=== c* and c/c* (ρ_solvent ≈ 1.0, will refine in Cell 3) ===")
print(f"  c* range: {df['c_star_wt_pct'].min():.2f} – {df['c_star_wt_pct'].max():.2f} wt%")
print(f"  c/c* range: {df['c_over_cstar'].min():.3f} – {df['c_over_cstar'].max():.3f}")

print(f"\n  c/c* percentiles:")
for p in [5, 25, 50, 75, 95]:
    print(f"    P{p}: {df['c_over_cstar'].quantile(p/100):.3f}")

low_cc = df[df['c_over_cstar'] < 0.1]
high_cc = df[df['c_over_cstar'] > 20]
if len(low_cc) > 0:
    print(f"\n  ⚠ c/c* < 0.1 ({len(low_cc)} rows):")
    print(low_cc[['Paper_ID','Polymer_type','Polymer_conc_wt','intrinsic_viscosity_dLg','c_over_cstar']].to_string())
if len(high_cc) > 0:
    print(f"\n  ⚠ c/c* > 20 ({len(high_cc)} rows):")
    print(high_cc[['Paper_ID','Polymer_type','Polymer_conc_wt','intrinsic_viscosity_dLg','c_over_cstar']].to_string())

mh_df = pd.DataFrame([
    {'Polymer': k, 'K': v[0], 'a': v[1], 'Solvent_ref': v[2], 'Source': v[3]}
    for k, v in MH_PARAMS.items()
])
mh_df.to_csv(os.path.join(BASE, '0_Data/lookups/mark_houwink_parameters.csv'), index=False)
print(f"\n✓ Mark-Houwink lookup saved: mark_houwink_parameters.csv")

df.to_csv(os.path.join(BASE, '0_Data/processed/step2_intrinsic_viscosity.csv'), index=False)
print(f"✓ Checkpoint saved: step2_intrinsic_viscosity.csv ({df.shape[0]} rows)")
print("\n" + "="*60)
print("Cell 2 complete. Review [η] and c/c* values.")
print("Note: c/c* uses ρ≈1.0 — will be refined with actual")
print("solvent densities in Cell 3.")
print("="*60)

In [ ]:
SOLVENT_PROPS = {
    'chloroform':           {'ST': 27.14, 'BP': 61.2,  'rho': 1.489},
    'acetone':              {'ST': 23.46, 'BP': 56.1,  'rho': 0.784},
    'DCM':                  {'ST': 27.84, 'BP': 39.6,  'rho': 1.325},
    'DCE':                  {'ST': 31.86, 'BP': 83.5,  'rho': 1.253},
    'DMF':                  {'ST': 36.76, 'BP': 153.0, 'rho': 0.944},
    'THF':                  {'ST': 26.40, 'BP': 66.0,  'rho': 0.889},
    'TFE':                  {'ST': 21.08, 'BP': 74.0,  'rho': 1.393},
    'toluene':              {'ST': 28.52, 'BP': 110.6, 'rho': 0.867},
    'ethanol':              {'ST': 22.39, 'BP': 78.4,  'rho': 0.789},
    'methanol':             {'ST': 22.50, 'BP': 64.7,  'rho': 0.791},
    'water':                {'ST': 72.00, 'BP': 100.0, 'rho': 1.000},
    'acetic_acid':          {'ST': 27.10, 'BP': 118.1, 'rho': 1.049},
    'formic_acid':          {'ST': 37.67, 'BP': 100.8, 'rho': 1.220},
    'ethyl_acetate':        {'ST': 23.75, 'BP': 77.1,  'rho': 0.902},
    'isopropyl_alcohol':    {'ST': 21.22, 'BP': 82.6,  'rho': 0.786},
    'dimethyl_carbonate':   {'ST': 28.50, 'BP': 90.0,  'rho': 1.069},
    'tetrahydronaphthalene':{'ST': 33.08, 'BP': 207.6, 'rho': 0.970},
    'cyclohexanone':        {'ST': 34.57, 'BP': 155.7, 'rho': 0.947},
}

def parse_ratio(ratio_str, n_components=2):
    """Parse solvent ratio string → volume fractions."""
    s = str(ratio_str).strip().replace('_v/v','').replace(' v/v','').replace('v/v','')
    s = s.replace('/',':', 1) if '/' in s and ':' not in s else s
    parts = s.split(':')
    if len(parts) == n_components:
        try:
            vals = [float(p) for p in parts]
            total = sum(vals)
            return [v/total for v in vals]
        except:
            pass
    return None

def get_solvent_props(row):
    """Return ST, BP, density for a row based on solvent system + ratio."""
    solv = row['Solvent_system']
    ratio = row['Solvent_ratio']

    if solv in SOLVENT_PROPS:
        p = SOLVENT_PROPS[solv]
        return p['ST'], p['BP'], p['rho']

    if '/' in solv:
        parts = solv.split('/')
        if len(parts) == 2:
            s1, s2 = parts[0].strip(), parts[1].strip()

            p1 = SOLVENT_PROPS.get(s1)
            p2 = SOLVENT_PROPS.get(s2)

            if p1 is None or p2 is None:
                return np.nan, np.nan, np.nan

            fracs = parse_ratio(ratio)
            if fracs is None:
                fracs = [0.5, 0.5]

            f1, f2 = fracs

            ST = f1 * p1['ST'] + f2 * p2['ST']
            BP = f1 * p1['BP'] + f2 * p2['BP']
            rho = f1 * p1['rho'] + f2 * p2['rho']

            return ST, BP, rho

    return np.nan, np.nan, np.nan

results = df.apply(get_solvent_props, axis=1, result_type='expand')
results.columns = ['Solvent_ST_mNm', 'Solvent_BP_C', 'Solvent_density_gmL']
df = pd.concat([df, results], axis=1)

print("=== SOLVENT PROPERTY COVERAGE ===")
nan_st = df['Solvent_ST_mNm'].isna().sum()
nan_bp = df['Solvent_BP_C'].isna().sum()
nan_rho = df['Solvent_density_gmL'].isna().sum()
print(f"  Missing ST: {nan_st}  |  Missing BP: {nan_bp}  |  Missing ρ: {nan_rho}")

if nan_st > 0:
    missing = df[df['Solvent_ST_mNm'].isna()]
    print(f"\n  ⚠ Rows with missing properties:")
    print(missing[['Paper_ID','Solvent_system','Solvent_ratio']].drop_duplicates().to_string(index=False))

print(f"\n=== SOLVENT PROPERTIES BY SYSTEM ===")
solv_summary = df.groupby(['Solvent_system', 'Solvent_ratio']).agg(
    ST=('Solvent_ST_mNm', 'first'),
    BP=('Solvent_BP_C', 'first'),
    rho=('Solvent_density_gmL', 'first'),
    n=('Paper_ID', 'size')
).reset_index().sort_values('n', ascending=False)
print(solv_summary.to_string(index=False))

print(f"\n=== REFINING c/c* WITH ACTUAL SOLVENT DENSITY ===")
old_cc = df['c_over_cstar'].copy()

df['c_star_wt_pct'] = 10.0 / (df['intrinsic_viscosity_dLg'] * df['Solvent_density_gmL'])
df['c_over_cstar'] = df['Polymer_conc_wt'] / df['c_star_wt_pct']

print(f"  c/c* range (old, ρ=1.0):  {old_cc.min():.4f} – {old_cc.max():.4f}")
print(f"  c/c* range (new, actual ρ): {df['c_over_cstar'].min():.4f} – {df['c_over_cstar'].max():.4f}")

delta = (df['c_over_cstar'] - old_cc).abs()
biggest = delta.nlargest(5).index
print(f"\n  Largest c/c* changes (top 5):")
for idx in biggest:
    print(f"    {df.loc[idx,'Paper_ID']:20s}  {df.loc[idx,'Polymer_type']:10s}  "
          f"ρ={df.loc[idx,'Solvent_density_gmL']:.3f}  "
          f"old={old_cc[idx]:.4f}  new={df.loc[idx,'c_over_cstar']:.4f}")

print(f"\n=== FEATURE RANGE CHECKS ===")
for col, (lo, hi, unit) in {
    'Solvent_ST_mNm': (20, 75, 'mN/m'),
    'Solvent_BP_C':   (35, 210, '°C'),
    'Solvent_density_gmL': (0.7, 1.6, 'g/mL'),
}.items():
    cmin, cmax = df[col].min(), df[col].max()
    flag = "✓" if cmin >= lo and cmax <= hi else "⚠"
    print(f"  {flag} {col}: [{cmin:.2f}, {cmax:.2f}] {unit}")

solv_lookup = df[['Solvent_system','Solvent_ratio','Solvent_ST_mNm','Solvent_BP_C','Solvent_density_gmL']].drop_duplicates()
solv_lookup = solv_lookup.sort_values('Solvent_system')
solv_lookup.to_csv(os.path.join(BASE, '0_Data/lookups/solvent_properties.csv'), index=False)
print(f"\n✓ Solvent lookup saved: solvent_properties.csv ({len(solv_lookup)} entries)")

df.to_csv(os.path.join(BASE, '0_Data/processed/step3_solvent_properties.csv'), index=False)
print(f"✓ Checkpoint saved: step3_solvent_properties.csv ({df.shape[0]} rows)")
print("\n" + "="*60)
print("Cell 3 complete. Review solvent properties and c/c* changes.")
print("Key: any NaN rows must be resolved before Cell 4.")
print("="*60)

In [ ]:
print("=== FIX: formic_acid/dichloromethane ===")
mask_fa_dcm = (df['Solvent_system'] == 'formic_acid/dichloromethane')
print(f"  Rows: {mask_fa_dcm.sum()}")
print(f"  Ratio: {df.loc[mask_fa_dcm, 'Solvent_ratio'].unique()}")

f1, f2 = 0.6, 0.4
fa = SOLVENT_PROPS['formic_acid']
dcm = SOLVENT_PROPS['DCM']

ST_mix = f1 * fa['ST'] + f2 * dcm['ST']
BP_mix = f1 * fa['BP'] + f2 * dcm['BP']
rho_mix = f1 * fa['rho'] + f2 * dcm['rho']
print(f"  Computed: ST={ST_mix:.2f} mN/m, BP={BP_mix:.2f} °C, ρ={rho_mix:.3f} g/mL")

df.loc[mask_fa_dcm, 'Solvent_ST_mNm'] = ST_mix
df.loc[mask_fa_dcm, 'Solvent_BP_C'] = BP_mix
df.loc[mask_fa_dcm, 'Solvent_density_gmL'] = rho_mix

df.loc[mask_fa_dcm, 'c_star_wt_pct'] = 10.0 / (
    df.loc[mask_fa_dcm, 'intrinsic_viscosity_dLg'] * df.loc[mask_fa_dcm, 'Solvent_density_gmL']
)
df.loc[mask_fa_dcm, 'c_over_cstar'] = (
    df.loc[mask_fa_dcm, 'Polymer_conc_wt'] / df.loc[mask_fa_dcm, 'c_star_wt_pct']
)
print(f"  c/c* for these rows: {df.loc[mask_fa_dcm, 'c_over_cstar'].unique()}")

nan_remaining = df[['Solvent_ST_mNm','Solvent_BP_C','Solvent_density_gmL']].isna().sum().sum()
print(f"\n  Remaining NaN in solvent properties: {nan_remaining}")
assert nan_remaining == 0, "STILL HAVE NaN — fix before proceeding!"
print(f"  ✓ All 408 rows have complete solvent properties")

print("\n=== LOG-TRANSFORM TARGET ===")
df['log_diameter'] = np.log(df['Fiber_diameter_nm_std'])
print(f"  Fiber_diameter_nm_std: [{df['Fiber_diameter_nm_std'].min():.0f}, {df['Fiber_diameter_nm_std'].max():.0f}] nm")
print(f"  log(diameter):         [{df['log_diameter'].min():.4f}, {df['log_diameter'].max():.4f}]")

ML_FEATURES = [
    'Polymer_conc_wt',          # concentration (wt%)
    'intrinsic_viscosity_dLg',  # [η] (dL/g)
    'c_over_cstar',             # c/c*
    'Solvent_ST_mNm',           # solvent surface tension (mN/m)
    'Solvent_BP_C',             # solvent boiling point (°C)
    'Air_pressure_bar_std',     # air pressure (bar)
    'Feed_rate_mLh_std',        # feed rate (mL/h)
    'Working_distance_cm',      # working distance (cm)
]
TARGET = 'log_diameter'

META_COLS = [
    'Paper_ID', 'DOI', 'Journal', 'Year', 'Polymer_type',
    'Polymer_MW_kDa', 'Solvent_system', 'Solvent_ratio',
    'Fiber_diameter_nm_std', 'Solvent_density_gmL', 'c_star_wt_pct',
]

df_ml = df[META_COLS + ML_FEATURES + [TARGET]].copy()

print(f"\n=== ML-READY DATASET ===")
print(f"  Shape: {df_ml.shape}")
print(f"  Features: {ML_FEATURES}")
print(f"  Target: {TARGET}")
print(f"  Metadata: {META_COLS}")

print(f"\n=== FINAL DATA INTEGRITY ===")
for col in ML_FEATURES + [TARGET]:
    n_nan = df_ml[col].isna().sum()
    n_inf = np.isinf(df_ml[col]).sum()
    cmin, cmax = df_ml[col].min(), df_ml[col].max()
    flag = "✓" if (n_nan == 0 and n_inf == 0) else "⚠"
    print(f"  {flag} {col:30s}: [{cmin:10.4f}, {cmax:10.4f}]  NaN={n_nan}  Inf={n_inf}")

print(f"\n=== FEATURE STATISTICS ===")
print(df_ml[ML_FEATURES + [TARGET]].describe().round(4).to_string())

print(f"\n=== FEATURE CORRELATIONS ===")
corr = df_ml[ML_FEATURES + [TARGET]].corr()
print(corr.round(3).to_string())

print(f"\n=== DATASET COMPOSITION ===")
print(f"  Total rows: {len(df_ml)}")
print(f"  Papers: {df_ml['Paper_ID'].nunique()}")
print(f"  Polymers: {df_ml['Polymer_type'].nunique()}")
print(f"  Year range: {df_ml['Year'].min()} – {df_ml['Year'].max()}")
print(f"  Diameter range: {df_ml['Fiber_diameter_nm_std'].min():.0f} – {df_ml['Fiber_diameter_nm_std'].max():.0f} nm")

paper_counts = df_ml.groupby('Paper_ID').size()
big_papers = paper_counts[paper_counts >= 5].sort_values(ascending=False)
print(f"\n  Papers with ≥5 rows (holdout candidates): {len(big_papers)}")
print(big_papers.to_string())

ml_path = os.path.join(BASE, '0_Data/processed/dataset_ml_ready.csv')
df_ml.to_csv(ml_path, index=False)
print(f"\n✓ ML-ready CSV saved: dataset_ml_ready.csv ({df_ml.shape[0]} rows × {df_ml.shape[1]} cols)")

pub_cols = list(df.columns)
for drop_col in ['Solvent_system_raw']:
    if drop_col in pub_cols:
        pub_cols.remove(drop_col)
df_pub = df[pub_cols].copy()
pub_path = os.path.join(BASE, '0_Data/processed/dataset_full.csv')
df_pub.to_csv(pub_path, index=False)
print(f"✓ Publication CSV saved: dataset_full.csv ({df_pub.shape[0]} rows × {df_pub.shape[1]} cols)")

print("\n" + "="*60)
print("Cell 4 complete. ML-ready dataset assembled.")
print("Review feature stats and correlations before Cell 5.")
print("="*60)

In [ ]:
from sklearn.preprocessing import StandardScaler
import pickle

ML_FEATURES = [
    'Polymer_conc_wt', 'intrinsic_viscosity_dLg', 'c_over_cstar',
    'Solvent_ST_mNm', 'Solvent_BP_C', 'Air_pressure_bar_std',
    'Feed_rate_mLh_std', 'Working_distance_cm',
]

scaler_full = StandardScaler()
X_scaled = scaler_full.fit_transform(df_ml[ML_FEATURES])

print("=== STANDARD SCALER (full dataset reference) ===")
print(f"  Fit on {len(df_ml)} rows × {len(ML_FEATURES)} features")
print(f"\n  Feature means:")
for feat, mean in zip(ML_FEATURES, scaler_full.mean_):
    print(f"    {feat:30s}: {mean:.4f}")
print(f"\n  Feature stds:")
for feat, std in zip(ML_FEATURES, scaler_full.scale_):
    print(f"    {feat:30s}: {std:.4f}")

scaler_path = os.path.join(BASE, '0_Data/scalers/standard_scaler_full.pkl')
with open(scaler_path, 'wb') as f:
    pickle.dump(scaler_full, f)
print(f"\n✓ Scaler saved: standard_scaler_full.pkl")
print(f"  (Note: training scaler will be refit on CV set in Session 2)")

print(f"\n=== SCALED DATA VERIFICATION ===")
X_df = pd.DataFrame(X_scaled, columns=ML_FEATURES)
print(X_df.describe().round(4).to_string())

In [ ]:
import os, json, time, warnings, pickle
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime

warnings.filterwarnings('ignore')

ROOT = PROJECT_ROOT
DATA = ROOT / '0_Data'
PROCESSED = DATA / 'processed'
SCALERS = DATA / 'scalers'
MODELS = ROOT / '4_Models'
L1_RESULTS = ROOT / '1_Level1_CV' / 'results'
L1_FIGURES = ROOT / '1_Level1_CV' / 'figures'
L1_TABLES  = ROOT / '1_Level1_CV' / 'tables'

for d in [MODELS, L1_RESULTS, L1_FIGURES, L1_TABLES, SCALERS]:
    d.mkdir(parents=True, exist_ok=True)

FEATURES = [
    'Polymer_conc_wt', 'intrinsic_viscosity_dLg', 'c_over_cstar',
    'Solvent_ST_mNm', 'Solvent_BP_C',
    'Air_pressure_bar_std', 'Feed_rate_mLh_std', 'Working_distance_cm'
]
TARGET = 'log_diameter'
SEED = 42

HOLDOUT_PAPERS = [
    'Liu2018',             # Gelatin (FSG), 7 rows
    'Mogharbel2023',       # PC, 8 rows
    'Rabello2025',         # PHBV, 8 rows
    'Ramos2022',           # TPU, 5 rows
    'Pakolpakcil2024',     # PBS, 5 rows
    'Kuk2016',             # PU, 9 rows
    'Alvarenga2024',       # PA6, 8 rows
    'Ergun2021',           # PAN, 5 rows
    'Czarnecka2021',       # PCL, 7 rows
    'GonzalezBenito2024',  # PLA, 5 rows
]

ML_MODELS = ['ExtraTrees', 'RandomForest', 'XGBoost', 'LightGBM',
             'CatBoost', 'SVR', 'ElasticNet', 'KNN']
DL_MODELS = ['MLP', 'CNN1D', 'TabNet', 'FT_Transformer']
ALL_MODELS = ML_MODELS + DL_MODELS

def checkpoint(obj, name, folder=L1_RESULTS):
    """Save object to Drive with timestamp."""
    path = folder / f'{name}.pkl'
    with open(path, 'wb') as f:
        pickle.dump(obj, f)
    print(f'  ✓ Checkpoint saved: {path.name}')

def load_checkpoint(name, folder=L1_RESULTS):
    """Load checkpoint from Drive."""
    path = folder / f'{name}.pkl'
    if path.exists():
        with open(path, 'rb') as f:
            return pickle.load(f)
    return None

df = pd.read_csv(PROCESSED / 'dataset_ml_ready.csv')

print(f'Dataset loaded: {df.shape[0]} rows, {df.Paper_ID.nunique()} papers, '
      f'{df.Polymer_type.nunique()} polymers')
print(f'Features: {len(FEATURES)}')
print(f'Target: {TARGET}')
print(f'Holdout papers: {len(HOLDOUT_PAPERS)}')

In [ ]:
from sklearn.preprocessing import StandardScaler

mask_ho = df.Paper_ID.isin(HOLDOUT_PAPERS)
df_cv = df[~mask_ho].copy().reset_index(drop=True)
df_ho = df[mask_ho].copy().reset_index(drop=True)

cv_polymers = set(df_cv.Polymer_type.unique())
ho_polymers = set(df_ho.Polymer_type.unique())
unseen = ho_polymers - cv_polymers
shared = ho_polymers & cv_polymers

print('='*60)
print('HOLDOUT SPLIT SUMMARY')
print('='*60)
print(f'CV set:      {len(df_cv):3d} rows | {df_cv.Paper_ID.nunique():2d} papers | {len(cv_polymers):2d} polymers')
print(f'Holdout set: {len(df_ho):3d} rows | {df_ho.Paper_ID.nunique():2d} papers | {len(ho_polymers):2d} polymers')
print(f'Unseen polymers ({len(unseen)}): {sorted(unseen)}')
print(f'Shared polymers ({len(shared)}): {sorted(shared)}')

print(f'\nHoldout papers:')
for p in sorted(df_ho.Paper_ID.unique()):
    sub = df_ho[df_ho.Paper_ID == p]
    poly = sub.Polymer_type.iloc[0]
    tag = ' [UNSEEN]' if poly in unseen else ''
    print(f'  {p:30s} | {len(sub):2d} rows | {poly}{tag}')

print(f'\nFeature range coverage (holdout within CV range?):')
for f in FEATURES:
    cv_min, cv_max = df_cv[f].min(), df_cv[f].max()
    ho_min, ho_max = df_ho[f].min(), df_ho[f].max()
    within = 'YES' if ho_min >= cv_min and ho_max <= cv_max else 'PARTIAL'
    print(f'  {f:28s} | CV [{cv_min:8.2f}, {cv_max:8.2f}] | HO [{ho_min:8.2f}, {ho_max:8.2f}] | {within}')

scaler = StandardScaler()
X_cv = df_cv[FEATURES].values
y_cv = df_cv[TARGET].values
X_cv_scaled = scaler.fit_transform(X_cv)

X_ho = df_ho[FEATURES].values
y_ho = df_ho[TARGET].values
X_ho_scaled = scaler.transform(X_ho)

df_cv.to_csv(PROCESSED / 'cv_training_set.csv', index=False)
df_ho.to_csv(PROCESSED / 'holdout_set.csv', index=False)

with open(SCALERS / 'standard_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

split_info = {
    'seed': SEED,
    'holdout_papers': HOLDOUT_PAPERS,
    'cv_rows': len(df_cv),
    'cv_papers': df_cv.Paper_ID.nunique(),
    'cv_polymers': sorted(cv_polymers),
    'ho_rows': len(df_ho),
    'ho_papers': df_ho.Paper_ID.nunique(),
    'ho_polymers': sorted(ho_polymers),
    'unseen_polymers': sorted(unseen),
    'shared_polymers': sorted(shared),
    'scaler_mean': scaler.mean_.tolist(),
    'scaler_std': scaler.scale_.tolist(),
    'feature_names': FEATURES,
}
with open(PROCESSED / 'split_info.json', 'w') as f:
    json.dump(split_info, f, indent=2)

groups_cv = df_cv.Paper_ID.values  # for GroupKFold

checkpoint({'X_cv': X_cv_scaled, 'y_cv': y_cv,
            'X_ho': X_ho_scaled, 'y_ho': y_ho,
            'groups_cv': groups_cv,
            'split_info': split_info}, 'session2_split')

print(f'   Saved: cv_training_set.csv, holdout_set.csv, standard_scaler.pkl, split_info.json')
print(f'   CV shape: {X_cv_scaled.shape}, Holdout shape: {X_ho_scaled.shape}')

In [ ]:
import optuna
from sklearn.model_selection import KFold, cross_val_score
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.svm import SVR
from sklearn.linear_model import ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

optuna.logging.set_verbosity(optuna.logging.WARNING)

kf_reg = KFold(n_splits=5, shuffle=True, random_state=SEED)
N_TRIALS = 100

def get_objective(model_name):
    def objective(trial):
        if model_name == 'ExtraTrees':
            params = {
                'n_estimators': trial.suggest_int('n_estimators', 100, 1000, step=100),
                'max_depth': trial.suggest_int('max_depth', 5, 50),
                'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
                'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
                'max_features': trial.suggest_float('max_features', 0.3, 1.0),
                'random_state': SEED
            }
            model = ExtraTreesRegressor(**params)

        elif model_name == 'RandomForest':
            params = {
                'n_estimators': trial.suggest_int('n_estimators', 100, 1000, step=100),
                'max_depth': trial.suggest_int('max_depth', 5, 50),
                'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
                'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
                'max_features': trial.suggest_float('max_features', 0.3, 1.0),
                'random_state': SEED
            }
            model = RandomForestRegressor(**params)

        elif model_name == 'XGBoost':
            params = {
                'n_estimators': trial.suggest_int('n_estimators', 100, 1000, step=100),
                'max_depth': trial.suggest_int('max_depth', 3, 12),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
                'subsample': trial.suggest_float('subsample', 0.5, 1.0),
                'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
                'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
                'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
                'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
                'random_state': SEED, 'verbosity': 0
            }
            model = XGBRegressor(**params)

        elif model_name == 'LightGBM':
            params = {
                'n_estimators': trial.suggest_int('n_estimators', 100, 1000, step=100),
                'max_depth': trial.suggest_int('max_depth', 3, 12),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
                'subsample': trial.suggest_float('subsample', 0.5, 1.0),
                'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
                'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
                'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
                'num_leaves': trial.suggest_int('num_leaves', 15, 127),
                'min_child_samples': trial.suggest_int('min_child_samples', 5, 30),
                'random_state': SEED, 'verbose': -1
            }
            model = LGBMRegressor(**params)

        elif model_name == 'CatBoost':
            params = {
                'iterations': trial.suggest_int('iterations', 100, 1000, step=100),
                'depth': trial.suggest_int('depth', 4, 10),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
                'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-2, 10.0, log=True),
                'subsample': trial.suggest_float('subsample', 0.5, 1.0),
                'random_seed': SEED, 'verbose': 0
            }
            model = CatBoostRegressor(**params)

        elif model_name == 'SVR':
            params = {
                'C': trial.suggest_float('C', 0.1, 100.0, log=True),
                'epsilon': trial.suggest_float('epsilon', 0.01, 0.5, log=True),
                'gamma': trial.suggest_categorical('gamma', ['scale', 'auto']),
                'kernel': trial.suggest_categorical('kernel', ['rbf', 'poly']),
            }
            if params['kernel'] == 'poly':
                params['degree'] = trial.suggest_int('degree', 2, 4)
            model = SVR(**params)

        elif model_name == 'ElasticNet':
            params = {
                'alpha': trial.suggest_float('alpha', 1e-4, 10.0, log=True),
                'l1_ratio': trial.suggest_float('l1_ratio', 0.01, 1.0),
                'max_iter': 5000,
                'random_state': SEED
            }
            model = ElasticNet(**params)

        elif model_name == 'KNN':
            params = {
                'n_neighbors': trial.suggest_int('n_neighbors', 3, 30),
                'weights': trial.suggest_categorical('weights', ['uniform', 'distance']),
                'metric': trial.suggest_categorical('metric', ['euclidean', 'manhattan', 'minkowski']),
                'p': trial.suggest_int('p', 1, 3),
            }
            model = KNeighborsRegressor(**params)

        scores = cross_val_score(model, X_cv_scaled, y_cv,
                                 cv=kf_reg, scoring='r2', n_jobs=-1)
        return scores.mean()

    return objective

best_params_ml = load_checkpoint('best_params_ml') or {}
tuning_log = load_checkpoint('tuning_log_ml') or {}

print('='*60)
print('OPTUNA HP TUNING — 8 ML MODELS (100 trials each)')
print('='*60)

for name in ML_MODELS:
    if name in best_params_ml:
        print(f'\n⏭  {name}: already tuned (loaded from checkpoint), R²={tuning_log[name]["best_r2"]:.4f}')
        continue

    print(f'\n▶ Tuning {name}...')
    t0 = time.time()

    study = optuna.create_study(direction='maximize',
                                 sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(get_objective(name), n_trials=N_TRIALS, show_progress_bar=True)

    elapsed = time.time() - t0
    best_params_ml[name] = study.best_params
    tuning_log[name] = {
        'best_r2': study.best_value,
        'n_trials': N_TRIALS,
        'elapsed_sec': round(elapsed, 1)
    }

    print(f'  ✓ {name}: best R² = {study.best_value:.4f}  ({elapsed:.0f}s)')
    print(f'    Best params: {study.best_params}')

    checkpoint(best_params_ml, 'best_params_ml')
    checkpoint(tuning_log, 'tuning_log_ml')

print('\n' + '='*60)
print('TUNING SUMMARY')
print('='*60)
print(f'{"Model":15s} | {"Best R²":>8s} | {"Time (s)":>8s}')
print('-'*40)
for name in ML_MODELS:
    r2 = tuning_log[name]['best_r2']
    t = tuning_log[name]['elapsed_sec']
    print(f'{name:15s} | {r2:8.4f} | {t:8.1f}')

checkpoint(best_params_ml, 'best_params_ml')
checkpoint(tuning_log, 'tuning_log_ml')

In [ ]:
from sklearn.model_selection import KFold, GroupKFold, learning_curve
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import numpy as np

def build_model(name, params):
    p = params.copy()
    if name == 'ExtraTrees':
        p['random_state'] = SEED
        return ExtraTreesRegressor(**p)
    elif name == 'RandomForest':
        p['random_state'] = SEED
        return RandomForestRegressor(**p)
    elif name == 'XGBoost':
        p['random_state'] = SEED; p['verbosity'] = 0
        return XGBRegressor(**p)
    elif name == 'LightGBM':
        p['random_state'] = SEED; p['verbose'] = -1
        return LGBMRegressor(**p)
    elif name == 'CatBoost':
        p['random_seed'] = SEED; p['verbose'] = 0
        return CatBoostRegressor(**p)
    elif name == 'SVR':
        return SVR(**p)
    elif name == 'ElasticNet':
        p['random_state'] = SEED; p['max_iter'] = 5000
        return ElasticNet(**p)
    elif name == 'KNN':
        return KNeighborsRegressor(**p)

def compute_metrics(y_true, y_pred):
    """Compute metrics in log-scale and back-transform for MAPE."""
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    y_true_nm = np.exp(y_true)
    y_pred_nm = np.exp(y_pred)
    mape = np.mean(np.abs((y_true_nm - y_pred_nm) / y_true_nm)) * 100
    return {'R2': r2, 'RMSE': rmse, 'MAE': mae, 'MAPE_nm': mape}

kf_reg = KFold(n_splits=5, shuffle=True, random_state=SEED)
gkf = GroupKFold(n_splits=5)

cv_results = load_checkpoint('cv_results_ml') or {}
oof_predictions = load_checkpoint('oof_predictions_ml') or {}
lc_data = load_checkpoint('learning_curves_ml') or {}

print('='*60)
print('CV EVALUATION — 8 ML MODELS')
print('='*60)

for name in ML_MODELS:
    if name in cv_results:
        r2_reg = cv_results[name]['RegKFold']['R2_mean']
        r2_grp = cv_results[name]['GroupKFold']['R2_mean']
        print(f'\n⏭  {name}: loaded from checkpoint | RegKF R²={r2_reg:.4f} | GroupKF R²={r2_grp:.4f}')
        continue

    print(f'\n▶ {name}...')
    t0 = time.time()
    params = best_params_ml[name]

    reg_metrics = {'R2': [], 'RMSE': [], 'MAE': [], 'MAPE_nm': []}
    oof_pred = np.full(len(y_cv), np.nan)
    oof_fold = np.full(len(y_cv), -1, dtype=int)

    for fold_i, (train_idx, val_idx) in enumerate(kf_reg.split(X_cv_scaled)):
        model = build_model(name, params)
        model.fit(X_cv_scaled[train_idx], y_cv[train_idx])
        pred = model.predict(X_cv_scaled[val_idx])
        m = compute_metrics(y_cv[val_idx], pred)
        for k in reg_metrics:
            reg_metrics[k].append(m[k])
        oof_pred[val_idx] = pred
        oof_fold[val_idx] = fold_i

    reg_summary = {f'{k}_mean': np.mean(v) for k, v in reg_metrics.items()}
    reg_summary.update({f'{k}_std': np.std(v) for k, v in reg_metrics.items()})
    reg_summary['per_fold'] = {k: v for k, v in reg_metrics.items()}

    grp_metrics = {'R2': [], 'RMSE': [], 'MAE': [], 'MAPE_nm': []}
    for fold_i, (train_idx, val_idx) in enumerate(gkf.split(X_cv_scaled, y_cv, groups_cv)):
        model = build_model(name, params)
        model.fit(X_cv_scaled[train_idx], y_cv[train_idx])
        pred = model.predict(X_cv_scaled[val_idx])
        m = compute_metrics(y_cv[val_idx], pred)
        for k in grp_metrics:
            grp_metrics[k].append(m[k])

    grp_summary = {f'{k}_mean': np.mean(v) for k, v in grp_metrics.items()}
    grp_summary.update({f'{k}_std': np.std(v) for k, v in grp_metrics.items()})
    grp_summary['per_fold'] = {k: v for k, v in grp_metrics.items()}

    model_lc = build_model(name, params)
    train_sizes_abs, train_scores, val_scores = learning_curve(
        model_lc, X_cv_scaled, y_cv, cv=kf_reg,
        train_sizes=np.linspace(0.1, 1.0, 10),
        scoring='r2', n_jobs=-1, random_state=SEED
    )

    elapsed = time.time() - t0

    cv_results[name] = {
        'RegKFold': reg_summary,
        'GroupKFold': grp_summary,
        'elapsed_sec': round(elapsed, 1)
    }
    oof_predictions[name] = {
        'y_true': y_cv,
        'y_pred': oof_pred,
        'fold_idx': oof_fold
    }
    lc_data[name] = {
        'train_sizes': train_sizes_abs,
        'train_scores_mean': train_scores.mean(axis=1),
        'train_scores_std': train_scores.std(axis=1),
        'val_scores_mean': val_scores.mean(axis=1),
        'val_scores_std': val_scores.std(axis=1),
    }

    print(f'  RegKFold  — R²: {reg_summary["R2_mean"]:.4f} ± {reg_summary["R2_std"]:.4f}  |  '
          f'MAPE: {reg_summary["MAPE_nm_mean"]:.1f}% ± {reg_summary["MAPE_nm_std"]:.1f}%')
    print(f'  GroupKFold — R²: {grp_summary["R2_mean"]:.4f} ± {grp_summary["R2_std"]:.4f}  |  '
          f'MAPE: {grp_summary["MAPE_nm_mean"]:.1f}% ± {grp_summary["MAPE_nm_std"]:.1f}%')
    print(f'  OOF R²: {r2_score(y_cv, oof_pred):.4f}  ({elapsed:.0f}s)')

    checkpoint(cv_results, 'cv_results_ml')
    checkpoint(oof_predictions, 'oof_predictions_ml')
    checkpoint(lc_data, 'learning_curves_ml')

print('\n' + '='*70)
print('CV SUMMARY — ALL 8 ML MODELS')
print('='*70)
print(f'{"Model":15s} | {"RegKF R²":>12s} | {"GroupKF R²":>12s} | {"RegKF MAPE%":>12s} | {"OOF R²":>7s}')
print('-'*70)
for name in ML_MODELS:
    rr = cv_results[name]['RegKFold']
    gr = cv_results[name]['GroupKFold']
    oof_r2 = r2_score(oof_predictions[name]['y_true'], oof_predictions[name]['y_pred'])
    print(f'{name:15s} | {rr["R2_mean"]:.4f}±{rr["R2_std"]:.4f} | '
          f'{gr["R2_mean"]:.4f}±{gr["R2_std"]:.4f} | '
          f'{rr["MAPE_nm_mean"]:5.1f}±{rr["MAPE_nm_std"]:.1f}    | {oof_r2:.4f}')

print('\n▶ Training final models on full CV set...')
final_models_ml = {}
for name in ML_MODELS:
    model = build_model(name, best_params_ml[name])
    model.fit(X_cv_scaled, y_cv)
    final_models_ml[name] = model
    print(f'  ✓ {name} trained on full CV ({len(y_cv)} rows)')

checkpoint(final_models_ml, 'final_models_ml', folder=MODELS)

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from pytorch_tabnet.tab_model import TabNetRegressor

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device} ({torch.cuda.get_device_name() if torch.cuda.is_available() else "CPU"})')

kf_reg = KFold(n_splits=5, shuffle=True, random_state=SEED)
N_TRIALS_DL = 50

class MLPRegressor(nn.Module):
    def __init__(self, input_dim, hidden_dims, dropout):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)

class CNN1DRegressor(nn.Module):
    def __init__(self, input_dim, n_filters, kernel_size, fc_dim, dropout):
        super().__init__()
        self.conv1 = nn.Conv1d(1, n_filters, kernel_size, padding=kernel_size//2)
        self.relu = nn.ReLU()
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.dropout = nn.Dropout(dropout)
        self.fc1 = nn.Linear(n_filters, fc_dim)
        self.fc2 = nn.Linear(fc_dim, 1)

    def forward(self, x):
        x = x.unsqueeze(1)  # (batch, 1, features)
        x = self.relu(self.conv1(x))
        x = self.pool(x).squeeze(-1)
        x = self.dropout(x)
        x = self.relu(self.fc1(x))
        return self.fc2(x).squeeze(-1)

class FTTransformerRegressor(nn.Module):
    def __init__(self, input_dim, d_model, n_heads, n_layers, dropout):
        super().__init__()
        self.feature_embeddings = nn.ModuleList([
            nn.Linear(1, d_model) for _ in range(input_dim)
        ])
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model*2,
            dropout=dropout, batch_first=True, activation='gelu'
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.head = nn.Linear(d_model, 1)

    def forward(self, x):
        batch = x.size(0)
        tokens = [emb(x[:, i:i+1]) for i, emb in enumerate(self.feature_embeddings)]
        tokens = torch.stack(tokens, dim=1)  # (batch, n_features, d_model)
        cls = self.cls_token.expand(batch, -1, -1)
        tokens = torch.cat([cls, tokens], dim=1)
        out = self.transformer(tokens)
        return self.head(out[:, 0, :]).squeeze(-1)

def train_torch_model(model, X_train, y_train, X_val, y_val,
                      lr, batch_size, epochs, patience=15):
    """Train a PyTorch model with early stopping."""
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, patience=5, factor=0.5)
    criterion = nn.MSELoss()

    X_t = torch.FloatTensor(X_train).to(device)
    y_t = torch.FloatTensor(y_train).to(device)
    X_v = torch.FloatTensor(X_val).to(device)
    y_v = torch.FloatTensor(y_val).to(device)

    ds = TensorDataset(X_t, y_t)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=True)

    best_val_loss = float('inf')
    best_state = None
    wait = 0

    for epoch in range(epochs):
        model.train()
        for xb, yb in dl:
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(X_v), y_v).item()

        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        pred = model(X_v).cpu().numpy()
    return pred, model

def get_dl_objective(model_name):
    def objective(trial):
        fold_r2s = []

        for train_idx, val_idx in kf_reg.split(X_cv_scaled):
            X_tr, X_vl = X_cv_scaled[train_idx], X_cv_scaled[val_idx]
            y_tr, y_vl = y_cv[train_idx], y_cv[val_idx]

            if model_name == 'MLP':
                n_layers = trial.suggest_int('n_layers', 2, 4)
                hidden_dim = trial.suggest_categorical('hidden_dim', [32, 64, 128, 256])
                dropout = trial.suggest_float('dropout', 0.1, 0.5)
                lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
                batch_size = trial.suggest_categorical('batch_size', [16, 32, 64])

                hidden_dims = [hidden_dim] * n_layers
                model = MLPRegressor(8, hidden_dims, dropout)
                pred, _ = train_torch_model(model, X_tr, y_tr, X_vl, y_vl,
                                            lr, batch_size, epochs=200)

            elif model_name == 'CNN1D':
                n_filters = trial.suggest_categorical('n_filters', [16, 32, 64, 128])
                kernel_size = trial.suggest_categorical('kernel_size', [3, 5])
                fc_dim = trial.suggest_categorical('fc_dim', [32, 64, 128])
                dropout = trial.suggest_float('dropout', 0.1, 0.5)
                lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
                batch_size = trial.suggest_categorical('batch_size', [16, 32, 64])

                model = CNN1DRegressor(8, n_filters, kernel_size, fc_dim, dropout)
                pred, _ = train_torch_model(model, X_tr, y_tr, X_vl, y_vl,
                                            lr, batch_size, epochs=200)

            elif model_name == 'TabNet':
                n_d = trial.suggest_categorical('n_d', [8, 16, 32, 64])
                n_steps = trial.suggest_int('n_steps', 3, 7)
                gamma = trial.suggest_float('gamma', 1.0, 2.0)
                lr = trial.suggest_float('lr', 1e-3, 5e-2, log=True)
                mask_type = trial.suggest_categorical('mask_type', ['sparsemax', 'entmax'])

                model = TabNetRegressor(
                    n_d=n_d, n_a=n_d, n_steps=n_steps, gamma=gamma,
                    mask_type=mask_type, seed=SEED, verbose=0,
                    optimizer_params={'lr': lr},
                    scheduler_params={'step_size': 20, 'gamma': 0.9},
                    scheduler_fn=torch.optim.lr_scheduler.StepLR,
                    device_name=str(device)
                )
                model.fit(
                    X_tr, y_tr.reshape(-1, 1),
                    eval_set=[(X_vl, y_vl.reshape(-1, 1))],
                    max_epochs=200, patience=15, batch_size=64
                )
                pred = model.predict(X_vl).flatten()

            elif model_name == 'FT_Transformer':
                d_model = trial.suggest_categorical('d_model', [32, 64, 128])
                n_heads = trial.suggest_categorical('n_heads', [2, 4])
                n_layers = trial.suggest_int('n_layers', 1, 4)
                dropout = trial.suggest_float('dropout', 0.1, 0.4)
                lr = trial.suggest_float('lr', 1e-4, 5e-3, log=True)
                batch_size = trial.suggest_categorical('batch_size', [16, 32, 64])

                if d_model % n_heads != 0:
                    return float('-inf')

                model = FTTransformerRegressor(8, d_model, n_heads, n_layers, dropout)
                pred, _ = train_torch_model(model, X_tr, y_tr, X_vl, y_vl,
                                            lr, batch_size, epochs=200)

            fold_r2s.append(r2_score(y_vl, pred))

        return np.mean(fold_r2s)

    return objective

best_params_dl = load_checkpoint('best_params_dl') or {}
tuning_log_dl = load_checkpoint('tuning_log_dl') or {}

print('='*60)
print('OPTUNA HP TUNING — 4 DL MODELS (50 trials each)')
print('='*60)

for name in DL_MODELS:
    if name in best_params_dl:
        print(f'\n⏭  {name}: already tuned, R²={tuning_log_dl[name]["best_r2"]:.4f}')
        continue

    print(f'\n▶ Tuning {name}...')
    t0 = time.time()

    study = optuna.create_study(direction='maximize',
                                 sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(get_dl_objective(name), n_trials=N_TRIALS_DL,
                   show_progress_bar=True)

    elapsed = time.time() - t0
    best_params_dl[name] = study.best_params
    tuning_log_dl[name] = {
        'best_r2': study.best_value,
        'n_trials': N_TRIALS_DL,
        'elapsed_sec': round(elapsed, 1)
    }

    print(f'  ✓ {name}: best R² = {study.best_value:.4f}  ({elapsed:.0f}s)')
    print(f'    Best params: {study.best_params}')

    checkpoint(best_params_dl, 'best_params_dl')
    checkpoint(tuning_log_dl, 'tuning_log_dl')

print('\n' + '='*60)
print('DL TUNING SUMMARY')
print('='*60)
print(f'{"Model":15s} | {"Best R²":>8s} | {"Time (s)":>8s}')
print('-'*40)
for name in DL_MODELS:
    r2 = tuning_log_dl[name]['best_r2']
    t = tuning_log_dl[name]['elapsed_sec']
    print(f'{name:15s} | {r2:8.4f} | {t:8.1f}')

print('\n' + '='*60)
print('ALL 12 MODELS — TUNING R² (RegKFold)')
print('='*60)
all_tuning = {**tuning_log, **tuning_log_dl}
for name in ALL_MODELS:
    r2 = all_tuning[name]['best_r2']
    print(f'  {name:15s} | {r2:.4f}')

In [ ]:
tuning_log = load_checkpoint('tuning_log_ml')
all_tuning = {**tuning_log, **tuning_log_dl}
print('ALL 12 MODELS — TUNING R² (RegKFold)')
print('='*60)
for name in ALL_MODELS:
    r2 = all_tuning[name]['best_r2']
    print(f'  {name:15s} | {r2:.4f}')

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import KFold, GroupKFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from pytorch_tabnet.tab_model import TabNetRegressor
import numpy as np, time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class MLPRegressor(nn.Module):
    def __init__(self, input_dim, hidden_dims, dropout):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x).squeeze(-1)

class CNN1DRegressor(nn.Module):
    def __init__(self, input_dim, n_filters, kernel_size, fc_dim, dropout):
        super().__init__()
        self.conv1 = nn.Conv1d(1, n_filters, kernel_size, padding=kernel_size//2)
        self.relu = nn.ReLU()
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.dropout = nn.Dropout(dropout)
        self.fc1 = nn.Linear(n_filters, fc_dim)
        self.fc2 = nn.Linear(fc_dim, 1)
    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.relu(self.conv1(x))
        x = self.pool(x).squeeze(-1)
        x = self.dropout(x)
        x = self.relu(self.fc1(x))
        return self.fc2(x).squeeze(-1)

class FTTransformerRegressor(nn.Module):
    def __init__(self, input_dim, d_model, n_heads, n_layers, dropout):
        super().__init__()
        self.feature_embeddings = nn.ModuleList([
            nn.Linear(1, d_model) for _ in range(input_dim)
        ])
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model*2,
            dropout=dropout, batch_first=True, activation='gelu'
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.head = nn.Linear(d_model, 1)
    def forward(self, x):
        batch = x.size(0)
        tokens = [emb(x[:, i:i+1]) for i, emb in enumerate(self.feature_embeddings)]
        tokens = torch.stack(tokens, dim=1)
        cls = self.cls_token.expand(batch, -1, -1)
        tokens = torch.cat([cls, tokens], dim=1)
        out = self.transformer(tokens)
        return self.head(out[:, 0, :]).squeeze(-1)

def train_torch_model(model, X_train, y_train, X_val, y_val,
                      lr, batch_size, epochs, patience=15):
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)
    criterion = nn.MSELoss()
    X_t = torch.FloatTensor(X_train).to(device)
    y_t = torch.FloatTensor(y_train).to(device)
    X_v = torch.FloatTensor(X_val).to(device)
    y_v = torch.FloatTensor(y_val).to(device)
    ds = TensorDataset(X_t, y_t)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=True)
    best_val_loss = float('inf')
    best_state = None
    wait = 0
    for epoch in range(epochs):
        model.train()
        for xb, yb in dl:
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(X_v), y_v).item()
        scheduler.step(val_loss)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break
    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        pred = model(X_v).cpu().numpy()
    return pred, model

def build_dl_model(name, params):
    if name == 'MLP':
        hidden_dims = [params['hidden_dim']] * params['n_layers']
        return MLPRegressor(8, hidden_dims, params['dropout'])
    elif name == 'CNN1D':
        return CNN1DRegressor(8, params['n_filters'], params['kernel_size'],
                              params['fc_dim'], params['dropout'])
    elif name == 'FT_Transformer':
        return FTTransformerRegressor(8, params['d_model'], params['n_heads'],
                                      params['n_layers'], params['dropout'])
    return None  # TabNet handled separately

def compute_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    y_true_nm = np.exp(y_true)
    y_pred_nm = np.exp(y_pred)
    mape = np.mean(np.abs((y_true_nm - y_pred_nm) / y_true_nm)) * 100
    return {'R2': r2, 'RMSE': rmse, 'MAE': mae, 'MAPE_nm': mape}

best_params_dl = load_checkpoint('best_params_dl')
kf_reg = KFold(n_splits=5, shuffle=True, random_state=SEED)
gkf = GroupKFold(n_splits=5)

cv_results_dl = load_checkpoint('cv_results_dl') or {}
oof_predictions_dl = load_checkpoint('oof_predictions_dl') or {}
lc_data_dl = load_checkpoint('learning_curves_dl') or {}

print('='*60)
print('CV EVALUATION — 4 DL MODELS')
print('='*60)

for name in DL_MODELS:
    if name in cv_results_dl:
        r2_reg = cv_results_dl[name]['RegKFold']['R2_mean']
        r2_grp = cv_results_dl[name]['GroupKFold']['R2_mean']
        print(f'\n⏭  {name}: loaded | RegKF R²={r2_reg:.4f} | GroupKF R²={r2_grp:.4f}')
        continue

    print(f'\n▶ {name}...')
    t0 = time.time()
    params = best_params_dl[name]

    reg_metrics = {'R2': [], 'RMSE': [], 'MAE': [], 'MAPE_nm': []}
    oof_pred = np.full(len(y_cv), np.nan)
    oof_fold = np.full(len(y_cv), -1, dtype=int)

    for fold_i, (train_idx, val_idx) in enumerate(kf_reg.split(X_cv_scaled)):
        X_tr, X_vl = X_cv_scaled[train_idx], X_cv_scaled[val_idx]
        y_tr, y_vl = y_cv[train_idx], y_cv[val_idx]

        if name == 'TabNet':
            model = TabNetRegressor(
                n_d=params['n_d'], n_a=params['n_d'],
                n_steps=params['n_steps'], gamma=params['gamma'],
                mask_type=params['mask_type'], seed=SEED, verbose=0,
                optimizer_params={'lr': params['lr']},
                scheduler_params={'step_size': 20, 'gamma': 0.9},
                scheduler_fn=torch.optim.lr_scheduler.StepLR,
                device_name=str(device)
            )
            model.fit(X_tr, y_tr.reshape(-1, 1),
                      eval_set=[(X_vl, y_vl.reshape(-1, 1))],
                      max_epochs=200, patience=15, batch_size=64)
            pred = model.predict(X_vl).flatten()
        else:
            dl_model = build_dl_model(name, params)
            pred, _ = train_torch_model(dl_model, X_tr, y_tr, X_vl, y_vl,
                                         params['lr'], params['batch_size'], epochs=200)

        m = compute_metrics(y_vl, pred)
        for k in reg_metrics:
            reg_metrics[k].append(m[k])
        oof_pred[val_idx] = pred
        oof_fold[val_idx] = fold_i
        print(f'    Fold {fold_i+1}/5 — R²={m["R2"]:.4f}')

    reg_summary = {f'{k}_mean': np.mean(v) for k, v in reg_metrics.items()}
    reg_summary.update({f'{k}_std': np.std(v) for k, v in reg_metrics.items()})
    reg_summary['per_fold'] = {k: v for k, v in reg_metrics.items()}

    grp_metrics = {'R2': [], 'RMSE': [], 'MAE': [], 'MAPE_nm': []}
    for fold_i, (train_idx, val_idx) in enumerate(gkf.split(X_cv_scaled, y_cv, groups_cv)):
        X_tr, X_vl = X_cv_scaled[train_idx], X_cv_scaled[val_idx]
        y_tr, y_vl = y_cv[train_idx], y_cv[val_idx]

        if name == 'TabNet':
            model = TabNetRegressor(
                n_d=params['n_d'], n_a=params['n_d'],
                n_steps=params['n_steps'], gamma=params['gamma'],
                mask_type=params['mask_type'], seed=SEED, verbose=0,
                optimizer_params={'lr': params['lr']},
                scheduler_params={'step_size': 20, 'gamma': 0.9},
                scheduler_fn=torch.optim.lr_scheduler.StepLR,
                device_name=str(device)
            )
            model.fit(X_tr, y_tr.reshape(-1, 1),
                      eval_set=[(X_vl, y_vl.reshape(-1, 1))],
                      max_epochs=200, patience=15, batch_size=64)
            pred = model.predict(X_vl).flatten()
        else:
            dl_model = build_dl_model(name, params)
            pred, _ = train_torch_model(dl_model, X_tr, y_tr, X_vl, y_vl,
                                         params['lr'], params['batch_size'], epochs=200)

        m = compute_metrics(y_vl, pred)
        for k in grp_metrics:
            grp_metrics[k].append(m[k])

    grp_summary = {f'{k}_mean': np.mean(v) for k, v in grp_metrics.items()}
    grp_summary.update({f'{k}_std': np.std(v) for k, v in grp_metrics.items()})
    grp_summary['per_fold'] = {k: v for k, v in grp_metrics.items()}

    lc_train_sizes_frac = [0.2, 0.4, 0.6, 0.8, 1.0]
    lc_train_scores = []
    lc_val_scores = []
    n_cv = len(y_cv)

    for frac in lc_train_sizes_frac:
        fold_train_r2 = []
        fold_val_r2 = []
        for train_idx, val_idx in kf_reg.split(X_cv_scaled):
            n_use = max(10, int(len(train_idx) * frac))
            np.random.seed(SEED)
            sub_idx = np.random.choice(train_idx, n_use, replace=False)
            X_tr, y_tr = X_cv_scaled[sub_idx], y_cv[sub_idx]
            X_vl, y_vl = X_cv_scaled[val_idx], y_cv[val_idx]

            if name == 'TabNet':
                model = TabNetRegressor(
                    n_d=params['n_d'], n_a=params['n_d'],
                    n_steps=params['n_steps'], gamma=params['gamma'],
                    mask_type=params['mask_type'], seed=SEED, verbose=0,
                    optimizer_params={'lr': params['lr']},
                    scheduler_params={'step_size': 20, 'gamma': 0.9},
                    scheduler_fn=torch.optim.lr_scheduler.StepLR,
                    device_name=str(device)
                )
                model.fit(X_tr, y_tr.reshape(-1, 1),
                          eval_set=[(X_vl, y_vl.reshape(-1, 1))],
                          max_epochs=200, patience=15, batch_size=min(64, n_use))
                pred_tr = model.predict(X_tr).flatten()
                pred_vl = model.predict(X_vl).flatten()
            else:
                dl_model = build_dl_model(name, params)
                pred_vl, trained_model = train_torch_model(
                    dl_model, X_tr, y_tr, X_vl, y_vl,
                    params['lr'], min(params['batch_size'], n_use), epochs=200)
                trained_model.eval()
                with torch.no_grad():
                    pred_tr = trained_model(torch.FloatTensor(X_tr).to(device)).cpu().numpy()

            fold_train_r2.append(r2_score(y_tr, pred_tr))
            fold_val_r2.append(r2_score(y_vl, pred_vl))

        lc_train_scores.append(fold_train_r2)
        lc_val_scores.append(fold_val_r2)

    lc_train_arr = np.array(lc_train_scores)
    lc_val_arr = np.array(lc_val_scores)

    elapsed = time.time() - t0

    cv_results_dl[name] = {
        'RegKFold': reg_summary,
        'GroupKFold': grp_summary,
        'elapsed_sec': round(elapsed, 1)
    }
    oof_predictions_dl[name] = {
        'y_true': y_cv,
        'y_pred': oof_pred,
        'fold_idx': oof_fold
    }
    lc_data_dl[name] = {
        'train_sizes': [int(max(10, int(len(y_cv)*0.8*f))) for f in lc_train_sizes_frac],
        'train_scores_mean': lc_train_arr.mean(axis=1),
        'train_scores_std': lc_train_arr.std(axis=1),
        'val_scores_mean': lc_val_arr.mean(axis=1),
        'val_scores_std': lc_val_arr.std(axis=1),
    }

    print(f'  RegKFold  — R²: {reg_summary["R2_mean"]:.4f} ± {reg_summary["R2_std"]:.4f}  |  '
          f'MAPE: {reg_summary["MAPE_nm_mean"]:.1f}% ± {reg_summary["MAPE_nm_std"]:.1f}%')
    print(f'  GroupKFold — R²: {grp_summary["R2_mean"]:.4f} ± {grp_summary["R2_std"]:.4f}  |  '
          f'MAPE: {grp_summary["MAPE_nm_mean"]:.1f}% ± {grp_summary["MAPE_nm_std"]:.1f}%')
    print(f'  OOF R²: {r2_score(y_cv, oof_pred):.4f}  ({elapsed:.0f}s)')

    checkpoint(cv_results_dl, 'cv_results_dl')
    checkpoint(oof_predictions_dl, 'oof_predictions_dl')
    checkpoint(lc_data_dl, 'learning_curves_dl')

print('\n' + '='*70)
print('CV SUMMARY — ALL 4 DL MODELS')
print('='*70)
print(f'{"Model":15s} | {"RegKF R²":>12s} | {"GroupKF R²":>12s} | {"RegKF MAPE%":>12s} | {"OOF R²":>7s}')
print('-'*70)
for name in DL_MODELS:
    rr = cv_results_dl[name]['RegKFold']
    gr = cv_results_dl[name]['GroupKFold']
    oof_r2 = r2_score(oof_predictions_dl[name]['y_true'], oof_predictions_dl[name]['y_pred'])
    print(f'{name:15s} | {rr["R2_mean"]:.4f}±{rr["R2_std"]:.4f} | '
          f'{gr["R2_mean"]:.4f}±{gr["R2_std"]:.4f} | '
          f'{rr["MAPE_nm_mean"]:5.1f}±{rr["MAPE_nm_std"]:.1f}    | {oof_r2:.4f}')

print('\n▶ Training final DL models on full CV set...')
final_models_dl = {}
for name in DL_MODELS:
    params = best_params_dl[name]
    n_train = int(len(y_cv) * 0.8)
    idx = np.arange(len(y_cv))
    np.random.seed(SEED)
    np.random.shuffle(idx)
    tr_idx, vl_idx = idx[:n_train], idx[n_train:]

    if name == 'TabNet':
        model = TabNetRegressor(
            n_d=params['n_d'], n_a=params['n_d'],
            n_steps=params['n_steps'], gamma=params['gamma'],
            mask_type=params['mask_type'], seed=SEED, verbose=0,
            optimizer_params={'lr': params['lr']},
            scheduler_params={'step_size': 20, 'gamma': 0.9},
            scheduler_fn=torch.optim.lr_scheduler.StepLR,
            device_name=str(device)
        )
        model.fit(X_cv_scaled[tr_idx], y_cv[tr_idx].reshape(-1, 1),
                  eval_set=[(X_cv_scaled[vl_idx], y_cv[vl_idx].reshape(-1, 1))],
                  max_epochs=200, patience=15, batch_size=64)
        final_models_dl[name] = model
    else:
        dl_model = build_dl_model(name, params)
        _, trained = train_torch_model(dl_model,
                                        X_cv_scaled[tr_idx], y_cv[tr_idx],
                                        X_cv_scaled[vl_idx], y_cv[vl_idx],
                                        params['lr'], params['batch_size'], epochs=200)
        final_models_dl[name] = trained.cpu()

    print(f'  ✓ {name} trained on full CV')

checkpoint(final_models_dl, 'final_models_dl', folder=MODELS)

In [ ]:
import json
from sklearn.metrics import r2_score

cv_results_ml = load_checkpoint('cv_results_ml')
cv_results_dl = load_checkpoint('cv_results_dl')
oof_ml = load_checkpoint('oof_predictions_ml')
oof_dl = load_checkpoint('oof_predictions_dl')
lc_ml = load_checkpoint('learning_curves_ml')
lc_dl = load_checkpoint('learning_curves_dl')
best_params_ml = load_checkpoint('best_params_ml')
best_params_dl = load_checkpoint('best_params_dl')
tuning_log_ml = load_checkpoint('tuning_log_ml')
tuning_log_dl = load_checkpoint('tuning_log_dl')

cv_results_all = {**cv_results_ml, **cv_results_dl}
oof_all = {**oof_ml, **oof_dl}
lc_all = {**lc_ml, **lc_dl}
best_params_all = {**best_params_ml, **best_params_dl}
tuning_log_all = {**tuning_log_ml, **tuning_log_dl}

checkpoint(cv_results_all, 'cv_results_all')
checkpoint(oof_all, 'oof_predictions_all')
checkpoint(lc_all, 'learning_curves_all')
checkpoint(best_params_all, 'best_params_all')

print('='*90)
print('GRAND SUMMARY: ALL 12 MODELS')
print('='*90)
print(f'{"#":>2s} {"Model":15s} | {"Type":4s} | {"Optuna R²":>9s} | '
      f'{"RegKF R²":>14s} | {"GroupKF R²":>14s} | {"MAPE%":>10s} | {"OOF R²":>7s}')
print('-'*90)

rows_for_csv = []
for i, name in enumerate(ALL_MODELS):
    mtype = 'ML' if name in ML_MODELS else 'DL'
    tune_r2 = tuning_log_all[name]['best_r2']
    rr = cv_results_all[name]['RegKFold']
    gr = cv_results_all[name]['GroupKFold']
    oof_r2 = r2_score(oof_all[name]['y_true'], oof_all[name]['y_pred'])

    print(f'{i+1:2d} {name:15s} | {mtype:4s} | {tune_r2:9.4f} | '
          f'{rr["R2_mean"]:.4f} ± {rr["R2_std"]:.4f} | '
          f'{gr["R2_mean"]:.4f} ± {gr["R2_std"]:.4f} | '
          f'{rr["MAPE_nm_mean"]:5.1f}±{rr["MAPE_nm_std"]:.1f}  | {oof_r2:.4f}')

    rows_for_csv.append({
        'Model': name, 'Type': mtype,
        'Optuna_R2': round(tune_r2, 4),
        'RegKF_R2_mean': round(rr['R2_mean'], 4),
        'RegKF_R2_std': round(rr['R2_std'], 4),
        'GroupKF_R2_mean': round(gr['R2_mean'], 4),
        'GroupKF_R2_std': round(gr['R2_std'], 4),
        'RegKF_RMSE_mean': round(rr['RMSE_mean'], 4),
        'RegKF_RMSE_std': round(rr['RMSE_std'], 4),
        'RegKF_MAE_mean': round(rr['MAE_mean'], 4),
        'RegKF_MAE_std': round(rr['MAE_std'], 4),
        'RegKF_MAPE_mean': round(rr['MAPE_nm_mean'], 1),
        'RegKF_MAPE_std': round(rr['MAPE_nm_std'], 1),
        'GroupKF_MAPE_mean': round(gr['MAPE_nm_mean'], 1),
        'GroupKF_MAPE_std': round(gr['MAPE_nm_std'], 1),
        'OOF_R2': round(oof_r2, 4),
    })

import pandas as pd
df_results = pd.DataFrame(rows_for_csv)
csv_path = L1_TABLES / 'cv_results_all_12_models.csv'
df_results.to_csv(csv_path, index=False)
print(f'\n✓ Table saved: {csv_path.name}')

best_reg = max(ALL_MODELS, key=lambda n: cv_results_all[n]['RegKFold']['R2_mean'])
best_reg_r2 = cv_results_all[best_reg]['RegKFold']['R2_mean']

best_grp = max(ALL_MODELS, key=lambda n: cv_results_all[n]['GroupKFold']['R2_mean'])
best_grp_r2 = cv_results_all[best_grp]['GroupKFold']['R2_mean']

best_mape_name = min(ALL_MODELS, key=lambda n: cv_results_all[n]['RegKFold']['MAPE_nm_mean'])
best_mape = cv_results_all[best_mape_name]['RegKFold']['MAPE_nm_mean']

print(f'\n{"="*60}')
print('KEY FINDINGS')
print(f'{"="*60}')
print(f'Best RegKFold R²:   {best_reg} ({best_reg_r2:.4f})')
print(f'Best GroupKFold R²: {best_grp} ({best_grp_r2:.4f})')
print(f'Best MAPE:          {best_mape_name} ({best_mape:.1f}%)')
print(f'RegKF→GroupKF gap:  {best_reg_r2:.4f} → {best_grp_r2:.4f}  (Δ={best_reg_r2-best_grp_r2:.4f})')

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import torch

L2_RESULTS = ROOT / '2_Level2_Holdout' / 'results'
L2_FIGURES = ROOT / '2_Level2_Holdout' / 'figures'
L2_TABLES  = ROOT / '2_Level2_Holdout' / 'tables'
for d in [L2_RESULTS, L2_FIGURES, L2_TABLES]:
    d.mkdir(parents=True, exist_ok=True)

final_models_ml = load_checkpoint('final_models_ml', folder=MODELS)
final_models_dl = load_checkpoint('final_models_dl', folder=MODELS)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def compute_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    y_true_nm = np.exp(y_true)
    y_pred_nm = np.exp(y_pred)
    mape = np.mean(np.abs((y_true_nm - y_pred_nm) / y_true_nm)) * 100
    return {'R2': r2, 'RMSE': rmse, 'MAE': mae, 'MAPE_nm': mape}

zs_predictions = {}
zs_results = {}

print('='*70)
print('LEVEL 2 — ZERO-SHOT HOLDOUT PREDICTIONS (67 rows, 10 papers)')
print('='*70)
print(f'{"#":>2s} {"Model":15s} | {"Type":4s} | {"R²":>7s} | {"RMSE":>6s} | {"MAE":>6s} | {"MAPE%":>7s}')
print('-'*70)

for i, name in enumerate(ALL_MODELS):
    if name in ML_MODELS:
        model = final_models_ml[name]
        pred = model.predict(X_ho_scaled)
    else:
        model = final_models_dl[name]
        if name == 'TabNet':
            pred = model.predict(X_ho_scaled).flatten()
        else:
            model = model.to(device)
            model.eval()
            with torch.no_grad():
                pred = model(torch.FloatTensor(X_ho_scaled).to(device)).cpu().numpy()

    m = compute_metrics(y_ho, pred)
    zs_predictions[name] = pred
    zs_results[name] = m
    mtype = 'ML' if name in ML_MODELS else 'DL'

    print(f'{i+1:2d} {name:15s} | {mtype:4s} | {m["R2"]:7.4f} | {m["RMSE"]:6.4f} | '
          f'{m["MAE"]:6.4f} | {m["MAPE_nm"]:6.1f}%')

print(f'\n{"="*70}')
print('PER-PAPER ZERO-SHOT MAPE% (top 3 models)')
print(f'{"="*70}')

top3 = sorted(ALL_MODELS, key=lambda n: zs_results[n]['MAPE_nm'])[:3]
print(f'Top 3 models: {top3}')
print(f'\n{"Paper":32s} | n  | {top3[0]:>10s} | {top3[1]:>10s} | {top3[2]:>10s}')
print('-'*80)

ho_papers = sorted(df_ho.Paper_ID.unique())
unseen_polymers = set(df_ho.Polymer_type.unique()) - set(df_cv.Polymer_type.unique())

for p in ho_papers:
    mask = df_ho.Paper_ID == p
    n = mask.sum()
    poly = df_ho[mask].Polymer_type.iloc[0]
    tag = ' *' if poly in unseen_polymers else ''

    mapes = []
    for name in top3:
        y_t = y_ho[mask.values]
        y_p = zs_predictions[name][mask.values]
        m = np.mean(np.abs((np.exp(y_t) - np.exp(y_p)) / np.exp(y_t))) * 100
        mapes.append(f'{m:9.1f}%')

    print(f'{p:30s}{tag} | {n:2d} | {" | ".join(mapes)}')

print(f'\n* = unseen polymer')

checkpoint({'predictions': zs_predictions, 'results': zs_results}, 'zeroshot_holdout')

best_zs = min(ALL_MODELS, key=lambda n: zs_results[n]['MAPE_nm'])
print(f'\nBest zero-shot model: {best_zs} (MAPE={zs_results[best_zs]["MAPE_nm"]:.1f}%)')

In [ ]:
from sklearn.linear_model import LinearRegression

N_SPLITS_ML = 100
N_SPLITS_DL = 50

def calibrate_paper(y_true, y_pred_zs, k, n_splits, seed=42):
    """
    For a single paper: randomly pick k anchor points, fit linear correction,
    predict remaining n-k points. Repeat n_splits times.
    Returns list of MAPE values (one per split).
    """
    n = len(y_true)
    if k >= n:
        return []

    rng = np.random.RandomState(seed)
    mapes = []

    for s in range(n_splits):
        idx = rng.permutation(n)
        anchor_idx = idx[:k]
        test_idx = idx[k:]

        if len(test_idx) == 0:
            continue

        lr = LinearRegression()
        lr.fit(y_pred_zs[anchor_idx].reshape(-1, 1), y_true[anchor_idx])

        y_calibrated = lr.predict(y_pred_zs[test_idx].reshape(-1, 1))

        y_true_nm = np.exp(y_true[test_idx])
        y_cal_nm = np.exp(y_calibrated)
        mape = np.mean(np.abs((y_true_nm - y_cal_nm) / y_true_nm)) * 100
        mapes.append(mape)

    return mapes

calibration_results = load_checkpoint('calibration_results_l2') or {}

print('='*70)
print('LEVEL 2 — CALIBRATION (per-paper linear correction)')
print('='*70)

for name in ALL_MODELS:
    if name in calibration_results:
        print(f'⏭  {name}: loaded from checkpoint')
        continue

    n_splits = N_SPLITS_ML if name in ML_MODELS else N_SPLITS_DL
    mtype = 'ML' if name in ML_MODELS else 'DL'
    pred_all = zs_predictions[name]
    t0 = time.time()

    model_results = {}

    for paper in ho_papers:
        mask = (df_ho.Paper_ID == paper).values
        y_p = y_ho[mask]
        pred_p = pred_all[mask]
        n = len(y_p)

        paper_results = {'n': n, 'k_values': {}}
        max_k = n // 2

        for k in range(2, max_k + 1):
            mapes = calibrate_paper(y_p, pred_p, k, n_splits)
            if mapes:
                paper_results['k_values'][k] = {
                    'mape_mean': np.mean(mapes),
                    'mape_std': np.std(mapes),
                    'mape_median': np.median(mapes),
                    'n_test': n - k,
                }

        model_results[paper] = paper_results

    elapsed = time.time() - t0
    calibration_results[name] = model_results

    mapes_k2 = [model_results[p]['k_values'][2]['mape_median']
                for p in ho_papers if 2 in model_results[p]['k_values']]
    median_k2 = np.median(mapes_k2) if mapes_k2 else float('nan')

    print(f'  ✓ {name:15s} ({mtype}) | median MAPE@k=2: {median_k2:5.1f}% | {elapsed:.1f}s')
    checkpoint(calibration_results, 'calibration_results_l2')

print(f'\n{"="*80}')
print('ZERO-SHOT vs CALIBRATED — ALL 12 MODELS')
print(f'{"="*80}')
print(f'{"#":>2s} {"Model":15s} | {"ZS MAPE%":>9s} | {"Cal@k=2":>9s} | {"Best k":>6s} | {"Cal@best":>9s} | {"Δ MAPE":>7s}')
print('-'*75)

summary_rows = []
for i, name in enumerate(ALL_MODELS):
    zs_mape = zs_results[name]['MAPE_nm']

    all_k_mapes = {}
    for paper in ho_papers:
        pr = calibration_results[name][paper]
        for k, vals in pr['k_values'].items():
            if k not in all_k_mapes:
                all_k_mapes[k] = []
            all_k_mapes[k].append(vals['mape_median'])

    k_medians = {k: np.median(v) for k, v in all_k_mapes.items()}

    cal_k2 = k_medians.get(2, float('nan'))
    best_k = min(k_medians, key=k_medians.get) if k_medians else 2
    cal_best = k_medians.get(best_k, float('nan'))
    delta = zs_mape - cal_best

    print(f'{i+1:2d} {name:15s} | {zs_mape:8.1f}% | {cal_k2:8.1f}% | {best_k:>6d} | '
          f'{cal_best:8.1f}% | {delta:+6.1f}%')

    summary_rows.append({
        'Model': name,
        'Type': 'ML' if name in ML_MODELS else 'DL',
        'ZS_MAPE': round(zs_mape, 1),
        'Cal_k2_MAPE': round(cal_k2, 1),
        'Best_k': best_k,
        'Cal_best_MAPE': round(cal_best, 1),
        'Delta_MAPE': round(delta, 1),
    })

df_cal_summary = pd.DataFrame(summary_rows)
df_cal_summary.to_csv(L2_TABLES / 'holdout_zs_vs_calibrated.csv', index=False)
print(f'\n✓ Table saved: holdout_zs_vs_calibrated.csv')

print(f'\n{"="*70}')
print('PER-PAPER CALIBRATION CURVES — Top 3 calibrated models')
print(f'{"="*70}')

best3_cal = sorted(ALL_MODELS,
                    key=lambda n: min(
                        np.median([calibration_results[n][p]['k_values'][k]['mape_median']
                                   for p in ho_papers if k in calibration_results[n][p]['k_values']])
                        for k in calibration_results[n][ho_papers[0]]['k_values']
                    ))[:3]

print(f'Top 3: {best3_cal}\n')

for paper in ho_papers:
    n = calibration_results[best3_cal[0]][paper]['n']
    poly = df_ho[df_ho.Paper_ID == paper].Polymer_type.iloc[0]
    tag = ' [UNSEEN]' if poly in unseen_polymers else ''
    print(f'{paper} (n={n}, {poly}{tag}):')

    for name in best3_cal:
        pr = calibration_results[name][paper]
        k_vals = sorted(pr['k_values'].keys())
        curve = ' → '.join([f'k={k}:{pr["k_values"][k]["mape_median"]:.0f}%' for k in k_vals])
        print(f'  {name:15s}: {curve}')
    print()

checkpoint(calibration_results, 'calibration_results_l2')

In [ ]:
cal_curve_data = {}
for name in ALL_MODELS:
    model_curves = {}
    for paper in ho_papers:
        pr = calibration_results[name][paper]
        k_vals = sorted(pr['k_values'].keys())
        model_curves[paper] = {
            'k_values': k_vals,
            'mape_mean': [pr['k_values'][k]['mape_mean'] for k in k_vals],
            'mape_std': [pr['k_values'][k]['mape_std'] for k in k_vals],
            'mape_median': [pr['k_values'][k]['mape_median'] for k in k_vals],
            'n_total': pr['n'],
        }
    cal_curve_data[name] = model_curves

checkpoint(cal_curve_data, 'calibration_curves_l2')

best_zs_model = min(ALL_MODELS, key=lambda n: zs_results[n]['MAPE_nm'])
best_zs_mape = zs_results[best_zs_model]['MAPE_nm']

def get_cal_mape(name, k=2):
    mapes = []
    for paper in ho_papers:
        pr = calibration_results[name][paper]
        if k in pr['k_values']:
            mapes.append(pr['k_values'][k]['mape_median'])
    return np.median(mapes) if mapes else float('inf')

best_cal_model = min(ALL_MODELS, key=lambda n: get_cal_mape(n, k=2))
best_cal_mape = get_cal_mape(best_cal_model, k=2)

print('='*70)
print('KEY FINDINGS — LEVEL 2 HOLDOUT')
print('='*70)
print(f'\nBest zero-shot:    {best_zs_model} (MAPE = {best_zs_mape:.1f}%)')
print(f'Best calibrated:   {best_cal_model} (MAPE@k=2 = {best_cal_mape:.1f}%)')
print(f'Ranking reversal:  {"YES" if best_zs_model != best_cal_model else "NO"}')

print(f'\nZero-shot ranking:')
zs_ranking = sorted(ALL_MODELS, key=lambda n: zs_results[n]['MAPE_nm'])
for i, name in enumerate(zs_ranking):
    print(f'  {i+1:2d}. {name:15s}  MAPE={zs_results[name]["MAPE_nm"]:.1f}%')

print(f'\nCalibrated ranking (k=2):')
cal_ranking = sorted(ALL_MODELS, key=lambda n: get_cal_mape(n, k=2))
for i, name in enumerate(cal_ranking):
    print(f'  {i+1:2d}. {name:15s}  MAPE={get_cal_mape(name, k=2):.1f}%')

print(f'\n{"="*70}')
print('UNSEEN vs SEEN POLYMER PERFORMANCE (best calibrated model)')
print(f'{"="*70}')

for paper in ho_papers:
    mask = (df_ho.Paper_ID == paper).values
    poly = df_ho[mask].Polymer_type.iloc[0]
    n = mask.sum()
    is_unseen = poly in unseen_polymers

    y_t_nm = np.exp(y_ho[mask])
    y_p_nm = np.exp(zs_predictions[best_cal_model][mask])
    zs_mape_paper = np.mean(np.abs((y_t_nm - y_p_nm) / y_t_nm)) * 100

    pr = calibration_results[best_cal_model][paper]
    cal_mape_paper = pr['k_values'].get(2, {}).get('mape_median', float('nan'))

    tag = 'UNSEEN' if is_unseen else 'seen'
    print(f'  {paper:30s} | {poly:18s} | {tag:6s} | n={n:2d} | '
          f'ZS={zs_mape_paper:5.1f}% → Cal={cal_mape_paper:5.1f}%')

In [ ]:
import torch

L3_RESULTS = ROOT / '3_Level3_External' / 'results'
L3_FIGURES = ROOT / '3_Level3_External' / 'figures'
L3_TABLES  = ROOT / '3_Level3_External' / 'tables'
for d in [L3_RESULTS, L3_FIGURES, L3_TABLES]:
    d.mkdir(parents=True, exist_ok=True)

external_path = PROJECT_ROOT / '3_Level3_External' / 'external_validation_features.csv'
df_external = pd.read_csv(external_path)

col_map = {
    'Intrinsic_viscosity_dLg': 'intrinsic_viscosity_dLg',
    'Surface_tension_solvent_mNm': 'Solvent_ST_mNm',
    'Boiling_point_solvent_C': 'Solvent_BP_C',
}
df_external.rename(columns=col_map, inplace=True)

df_external['log_diameter'] = np.log(df_external['Fiber_diameter_nm_std'])

X_external = df_external[FEATURES].values
y_external = df_external['log_diameter'].values

X_external_scaled = scaler.transform(X_external)

external_drive_path = DATA / 'processed' / 'external_validation.csv'
df_external.to_csv(external_drive_path, index=False)

print('='*70)
print('EXTERNAL VALIDATION DATA — Level 3')
print('='*70)
print(f'Rows: {len(df_external)}')
print(f'Polymer: PEO 1000 kDa')
print(f'Concentrations: {sorted(df_external.Polymer_conc_wt.unique())} wt%')
print(f'Pressure range: {df_external.Air_pressure_bar_std.min():.1f} – {df_external.Air_pressure_bar_std.max():.1f} bar')
print(f'Diameter range: {df_external.Fiber_diameter_nm_std.min():.0f} – {df_external.Fiber_diameter_nm_std.max():.0f} nm')
print(f'Log diameter range: {y_external.min():.3f} – {y_external.max():.3f}')

print(f'\nFeature range coverage (external set within CV range?):')
for f in FEATURES:
    cv_min, cv_max = df_cv[f].min(), df_cv[f].max()
    s_min, s_max = df_external[f].min(), df_external[f].max()
    within = 'YES' if s_min >= cv_min and s_max <= cv_max else 'PARTIAL'
    print(f'  {f:28s} | CV [{cv_min:8.2f}, {cv_max:8.2f}] | Ext  [{s_min:8.2f}, {s_max:8.2f}] | {within}')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

zs_external_predictions = {}
zs_external_results = {}

print(f'\n{"="*70}')
print('LEVEL 3 — ZERO-SHOT PREDICTIONS (21 rows)')
print(f'{"="*70}')
print(f'{"#":>2s} {"Model":15s} | {"Type":4s} | {"R²":>7s} | {"RMSE":>6s} | {"MAE":>6s} | {"MAPE%":>7s}')
print('-'*70)

for i, name in enumerate(ALL_MODELS):
    if name in ML_MODELS:
        model = final_models_ml[name]
        pred = model.predict(X_external_scaled)
    else:
        model = final_models_dl[name]
        if name == 'TabNet':
            pred = model.predict(X_external_scaled).flatten()
        else:
            model = model.to(device)
            model.eval()
            with torch.no_grad():
                pred = model(torch.FloatTensor(X_external_scaled).to(device)).cpu().numpy()

    m = compute_metrics(y_external, pred)
    zs_external_predictions[name] = pred
    zs_external_results[name] = m
    mtype = 'ML' if name in ML_MODELS else 'DL'

    print(f'{i+1:2d} {name:15s} | {mtype:4s} | {m["R2"]:7.4f} | {m["RMSE"]:6.4f} | '
          f'{m["MAE"]:6.4f} | {m["MAPE_nm"]:6.1f}%')

checkpoint({'predictions': zs_external_predictions, 'results': zs_external_results,
            'X_scaled': X_external_scaled, 'y': y_external}, 'zeroshot_external')

best_zs_external = min(ALL_MODELS, key=lambda n: zs_external_results[n]['MAPE_nm'])
print(f'\nBest zero-shot: {best_zs_external} (MAPE={zs_external_results[best_zs_external]["MAPE_nm"]:.1f}%)')

In [ ]:
from sklearn.linear_model import LinearRegression

N_SPLITS_L3 = 200

def calibrate_external(y_true, y_pred_zs, k, n_splits, seed=42):
    """Calibrate on external data: k anchors, predict rest, repeat n_splits times."""
    n = len(y_true)
    if k >= n:
        return []
    rng = np.random.RandomState(seed)
    mapes = []
    for s in range(n_splits):
        idx = rng.permutation(n)
        anchor_idx = idx[:k]
        test_idx = idx[k:]
        if len(test_idx) == 0:
            continue
        lr = LinearRegression()
        lr.fit(y_pred_zs[anchor_idx].reshape(-1, 1), y_true[anchor_idx])
        y_cal = lr.predict(y_pred_zs[test_idx].reshape(-1, 1))
        y_true_nm = np.exp(y_true[test_idx])
        y_cal_nm = np.exp(y_cal)
        mape = np.mean(np.abs((y_true_nm - y_cal_nm) / y_true_nm)) * 100
        mapes.append(mape)
    return mapes

cal_external_results = load_checkpoint('calibration_results_l3') or {}
K_VALUES = list(range(2, 11))  # k=2 to k=10

print('='*70)
print('LEVEL 3 — CALIBRATION (external, 200 splits per k)')
print('='*70)

for name in ALL_MODELS:
    if name in cal_external_results:
        print(f'⏭  {name}: loaded from checkpoint')
        continue

    t0 = time.time()
    pred = zs_external_predictions[name]
    model_cal = {}

    for k in K_VALUES:
        mapes = calibrate_external(y_external, pred, k, N_SPLITS_L3)
        if mapes:
            model_cal[k] = {
                'mapes': mapes,
                'mape_mean': np.mean(mapes),
                'mape_std': np.std(mapes),
                'mape_median': np.median(mapes),
                'n_test': len(y_external) - k,
            }

    elapsed = time.time() - t0
    cal_external_results[name] = model_cal

    mtype = 'ML' if name in ML_MODELS else 'DL'
    k5_mape = model_cal.get(5, {}).get('mape_median', float('nan'))
    print(f'  ✓ {name:15s} ({mtype}) | MAPE@k=5: {k5_mape:5.1f}% | {elapsed:.1f}s')
    checkpoint(cal_external_results, 'calibration_results_l3')

print(f'\n{"="*80}')
print('ZERO-SHOT vs CALIBRATED — ALL 12 MODELS (external)')
print(f'{"="*80}')
header = f'{"#":>2s} {"Model":15s} | {"ZS MAPE":>8s}'
for k in [2, 3, 5, 7, 10]:
    header += f' | {"k="+str(k):>8s}'
print(header)
print('-'*85)

for i, name in enumerate(ALL_MODELS):
    zs = zs_external_results[name]['MAPE_nm']
    row = f'{i+1:2d} {name:15s} | {zs:7.1f}%'
    for k in [2, 3, 5, 7, 10]:
        cal = cal_external_results[name].get(k, {}).get('mape_median', float('nan'))
        row += f' | {cal:7.1f}%'
    print(row)

print(f'\n{"="*70}')
print('CALIBRATION DISTRIBUTION @ k=5 (200 splits)')
print(f'{"="*70}')
print(f'{"Model":15s} | {"Median":>7s} | {"Mean":>7s} | {"SD":>6s} | {"Q25":>6s} | {"Q75":>6s}')
print('-'*60)

for name in ALL_MODELS:
    d = cal_external_results[name].get(5, {})
    if 'mapes' in d:
        mapes = np.array(d['mapes'])
        print(f'{name:15s} | {np.median(mapes):6.1f}% | {np.mean(mapes):6.1f}% | '
              f'{np.std(mapes):5.1f}% | {np.percentile(mapes, 25):5.1f}% | '
              f'{np.percentile(mapes, 75):5.1f}%')

print(f'\n{"="*70}')
print('BOOTSTRAP WIN COUNT @ k=5 (which model has lowest MAPE per split)')
print(f'{"="*70}')

win_counts = {name: 0 for name in ALL_MODELS}
for s in range(N_SPLITS_L3):
    best_mape = float('inf')
    best_model = None
    for name in ALL_MODELS:
        d = cal_external_results[name].get(5, {})
        if 'mapes' in d and s < len(d['mapes']):
            if d['mapes'][s] < best_mape:
                best_mape = d['mapes'][s]
                best_model = name
    if best_model:
        win_counts[best_model] += 1

for name in sorted(win_counts, key=win_counts.get, reverse=True):
    if win_counts[name] > 0:
        pct = win_counts[name] / N_SPLITS_L3 * 100
        print(f'  {name:15s}: {win_counts[name]:3d}/{N_SPLITS_L3} ({pct:.0f}%)')

checkpoint(cal_external_results, 'calibration_results_l3')
checkpoint(win_counts, 'win_counts_l3')

In [ ]:
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
import pingouin as pg

best_cal_external = min(ALL_MODELS,
                     key=lambda n: cal_external_results[n].get(5, {}).get('mape_median', float('inf')))
best_cal_mape_external = cal_external_results[best_cal_external][5]['mape_median']

print(f'Best calibrated model @ k=5: {best_cal_external} (median MAPE={best_cal_mape_external:.1f}%)')

mapes_list = cal_external_results[best_cal_external][5]['mapes']
median_idx = np.argsort(mapes_list)[len(mapes_list) // 2]

rng = np.random.RandomState(42)
for s in range(median_idx + 1):
    idx = rng.permutation(len(y_external))
anchor_idx = idx[:5]
test_idx = idx[5:]

pred_zs = zs_external_predictions[best_cal_external]
lr = LinearRegression()
lr.fit(pred_zs[anchor_idx].reshape(-1, 1), y_external[anchor_idx])
y_cal_log = lr.predict(pred_zs[test_idx].reshape(-1, 1))

y_test_log = y_external[test_idx]
y_test_nm = np.exp(y_test_log)
y_cal_nm = np.exp(y_cal_log)
y_zs_nm = np.exp(pred_zs[test_idx])

print(f'Anchor samples: {len(anchor_idx)}, Test samples: {len(test_idx)}')
print(f'Calibration slope: {lr.coef_[0]:.4f}, intercept: {lr.intercept_:.4f}')

print(f'\n{"="*70}')
print(f'11 STATISTICAL TESTS — {best_cal_external} (calibrated, k=5)')
print(f'{"="*70}')

stat_results = {}

ks_stat, ks_p = stats.ks_2samp(y_test_log, y_cal_log)
stat_results['KS_test'] = {'statistic': ks_stat, 'p_value': ks_p}
print(f' 1. KS test:              D={ks_stat:.4f}, p={ks_p:.4f}')

mw_stat, mw_p = stats.mannwhitneyu(y_test_nm, y_cal_nm, alternative='two-sided')
stat_results['Mann_Whitney'] = {'statistic': mw_stat, 'p_value': mw_p}
print(f' 2. Mann-Whitney U:       U={mw_stat:.1f}, p={mw_p:.4f}')

ba_mean = (y_test_nm + y_cal_nm) / 2
ba_diff = y_test_nm - y_cal_nm
ba_bias = np.mean(ba_diff)
ba_loa_lower = ba_bias - 1.96 * np.std(ba_diff)
ba_loa_upper = ba_bias + 1.96 * np.std(ba_diff)
stat_results['Bland_Altman'] = {'bias': ba_bias, 'LoA_lower': ba_loa_lower, 'LoA_upper': ba_loa_upper}
print(f' 3. Bland-Altman:         bias={ba_bias:.1f} nm, LoA=[{ba_loa_lower:.1f}, {ba_loa_upper:.1f}]')

bins = np.linspace(min(y_test_nm.min(), y_cal_nm.min()),
                   max(y_test_nm.max(), y_cal_nm.max()), 20)
p_hist = np.histogram(y_test_nm, bins=bins, density=True)[0] + 1e-10
q_hist = np.histogram(y_cal_nm, bins=bins, density=True)[0] + 1e-10
p_hist = p_hist / p_hist.sum()
q_hist = q_hist / q_hist.sum()
kl_div = stats.entropy(p_hist, q_hist)
stat_results['KL_divergence'] = {'value': kl_div}
print(f' 4. KL divergence:        {kl_div:.4f}')

wd = stats.wasserstein_distance(y_test_nm, y_cal_nm)
stat_results['Wasserstein'] = {'distance': wd}
print(f' 5. Wasserstein distance: {wd:.1f} nm')

from scipy.stats import gaussian_kde
try:
    kde_true = gaussian_kde(y_test_nm)
    kde_pred = gaussian_kde(y_cal_nm)
    x_grid = np.linspace(min(y_test_nm.min(), y_cal_nm.min()),
                         max(y_test_nm.max(), y_cal_nm.max()), 1000)
    overlap = np.trapz(np.minimum(kde_true(x_grid), kde_pred(x_grid)), x_grid)
    stat_results['Overlap_coefficient'] = {'value': overlap}
    print(f' 6. Overlap coefficient:  {overlap:.4f}')
except:
    stat_results['Overlap_coefficient'] = {'value': float('nan')}
    print(f' 6. Overlap coefficient:  failed (too few points)')

mean_true = np.mean(y_test_nm)
mean_pred = np.mean(y_cal_nm)
var_true = np.var(y_test_nm)
var_pred = np.var(y_cal_nm)
covar = np.mean((y_test_nm - mean_true) * (y_cal_nm - mean_pred))
ccc = (2 * covar) / (var_true + var_pred + (mean_true - mean_pred)**2)
stat_results['CCC'] = {'value': ccc}
print(f' 7. CCC:                  {ccc:.4f}')

icc_df = pd.DataFrame({
    'targets': np.concatenate([np.arange(len(test_idx)), np.arange(len(test_idx))]),
    'raters': ['actual'] * len(test_idx) + ['predicted'] * len(test_idx),
    'ratings': np.concatenate([y_test_nm, y_cal_nm])
})
try:
    icc_result = pg.intraclass_corr(data=icc_df, targets='targets',
                                     raters='raters', ratings='ratings')
    icc_3_1 = icc_result[icc_result['Type'] == 'ICC3']['ICC'].values[0]
    stat_results['ICC'] = {'ICC3': icc_3_1}
    print(f' 8. ICC (3,1):            {icc_3_1:.4f}')
except Exception as e:
    stat_results['ICC'] = {'error': str(e)}
    print(f' 8. ICC:                  failed ({e})')

r_pearson, p_pearson = stats.pearsonr(y_test_nm, y_cal_nm)
stat_results['Pearson'] = {'r': r_pearson, 'p_value': p_pearson}
print(f' 9. Pearson r:            {r_pearson:.4f}, p={p_pearson:.4f}')

r_spearman, p_spearman = stats.spearmanr(y_test_nm, y_cal_nm)
stat_results['Spearman'] = {'rho': r_spearman, 'p_value': p_spearman}
print(f'10. Spearman ρ:           {r_spearman:.4f}, p={p_spearman:.4f}')

t_stat, t_p = stats.ttest_rel(y_test_nm, y_cal_nm)
stat_results['Paired_ttest'] = {'t_statistic': t_stat, 'p_value': t_p}
print(f'11. Paired t-test:        t={t_stat:.4f}, p={t_p:.4f}')

mape_cal = np.mean(np.abs((y_test_nm - y_cal_nm) / y_test_nm)) * 100
r2_cal = r2_score(y_test_log, y_cal_log)
rmse_cal = np.sqrt(mean_squared_error(y_test_log, y_cal_log))

mape_zs = np.mean(np.abs((y_test_nm - y_zs_nm) / y_test_nm)) * 100
r2_zs = r2_score(y_test_log, pred_zs[test_idx])

print(f'\nCalibrated: R²={r2_cal:.4f}, MAPE={mape_cal:.1f}%')
print(f'Zero-shot:  R²={r2_zs:.4f}, MAPE={mape_zs:.1f}%')

checkpoint(stat_results, 'statistical_tests_l3')

stat_rows = []
for test_name, vals in stat_results.items():
    row = {'Test': test_name}
    row.update(vals)
    stat_rows.append(row)
pd.DataFrame(stat_rows).to_csv(L3_TABLES / 'statistical_tests.csv', index=False)
print(f'\n✓ Table saved: statistical_tests.csv')

best_zs_external = min(ALL_MODELS, key=lambda n: zs_external_results[n]['MAPE_nm'])
best_cal_k5 = best_cal_external

In [ ]:
import pingouin as pg
from scipy import stats

L1_ANALYSIS = ROOT / '1_Level1_CV' / 'results'

papers = df_cv.Paper_ID.values
y_log = df_cv.log_diameter.values

paper_ids = df_cv.Paper_ID.unique()
n_papers = len(paper_ids)
n_total = len(df_cv)

group_means = df_cv.groupby('Paper_ID')['log_diameter'].mean()
group_sizes = df_cv.groupby('Paper_ID')['log_diameter'].count()
grand_mean = y_log.mean()

SS_between = sum(group_sizes[p] * (group_means[p] - grand_mean)**2 for p in paper_ids)
df_between = n_papers - 1

SS_within = sum(((df_cv[df_cv.Paper_ID == p]['log_diameter'] - group_means[p])**2).sum()
                for p in paper_ids)
df_within = n_total - n_papers

MS_between = SS_between / df_between
MS_within = SS_within / df_within

n_0 = (n_total - sum(group_sizes**2) / n_total) / (n_papers - 1)

sigma2_between = (MS_between - MS_within) / n_0
sigma2_within = MS_within
sigma2_total = sigma2_between + sigma2_within

ICC_value = sigma2_between / sigma2_total

F_stat = MS_between / MS_within
F_p = 1 - stats.f.cdf(F_stat, df_between, df_within)

print('='*70)
print('ICC ANALYSIS — BETWEEN-PAPER VARIANCE')
print('='*70)
print(f'\nDataset: CV set ({n_total} rows, {n_papers} papers)')
print(f'Target: log(fiber diameter)')
print(f'\nVariance decomposition:')
print(f'  σ²_between (paper-to-paper): {sigma2_between:.4f} ({sigma2_between/sigma2_total*100:.1f}%)')
print(f'  σ²_within  (within-paper):   {sigma2_within:.4f} ({sigma2_within/sigma2_total*100:.1f}%)')
print(f'  σ²_total:                     {sigma2_total:.4f}')
print(f'\n  ICC = σ²_between / σ²_total = {ICC_value:.4f}')
print(f'  F-test: F={F_stat:.2f}, p={F_p:.2e}')
print(f'\nInterpretation: {ICC_value*100:.1f}% of fiber diameter variance is')
print(f'attributable to between-paper (lab-to-lab) effects.')

print(f'\n{"="*70}')
print('ICC PER FEATURE — Between-Paper Variance by Feature')
print(f'{"="*70}')
print(f'{"Feature":28s} | {"ICC":>6s} | {"% between":>10s} | {"F-stat":>7s}')
print('-'*60)

feature_iccs = {}
for feat in FEATURES + ['log_diameter']:
    vals = df_cv[feat].values
    g_means = df_cv.groupby('Paper_ID')[feat].mean()
    g_sizes = df_cv.groupby('Paper_ID')[feat].count()
    g_mean = vals.mean()

    ss_b = sum(g_sizes[p] * (g_means[p] - g_mean)**2 for p in paper_ids)
    ss_w = sum(((df_cv[df_cv.Paper_ID == p][feat] - g_means[p])**2).sum()
               for p in paper_ids)

    ms_b = ss_b / df_between
    ms_w = ss_w / df_within if df_within > 0 else 1e-10

    n0 = (n_total - sum(g_sizes**2) / n_total) / (n_papers - 1)
    s2_b = max(0, (ms_b - ms_w) / n0)
    s2_w = ms_w
    s2_t = s2_b + s2_w

    icc_f = s2_b / s2_t if s2_t > 0 else 0
    f_f = ms_b / ms_w if ms_w > 0 else 0

    feature_iccs[feat] = {'ICC': icc_f, 'pct_between': s2_b/s2_t*100,
                           'F': f_f, 'sigma2_between': s2_b, 'sigma2_within': s2_w}

    print(f'{feat:28s} | {icc_f:6.4f} | {s2_b/s2_t*100:9.1f}% | {f_f:7.2f}')

print(f'\n{"="*70}')
print('PER-PAPER STATISTICS')
print(f'{"="*70}')
print(f'{"Paper":30s} | {"n":>3s} | {"Mean(nm)":>9s} | {"SD(nm)":>8s} | {"CV%":>6s} | {"Polymer":>15s}')
print('-'*85)

paper_stats = []
for p in sorted(paper_ids):
    sub = df_cv[df_cv.Paper_ID == p]
    diams = np.exp(sub.log_diameter.values)
    poly = sub.Polymer_type.iloc[0]
    mn = diams.mean()
    sd = diams.std()
    cv = sd / mn * 100 if mn > 0 else 0
    print(f'{p:30s} | {len(sub):3d} | {mn:9.1f} | {sd:8.1f} | {cv:5.1f}% | {poly:>15s}')
    paper_stats.append({'Paper': p, 'n': len(sub), 'mean_nm': mn,
                        'sd_nm': sd, 'cv_pct': cv, 'polymer': poly})

df_paper_stats = pd.DataFrame(paper_stats)
df_paper_stats.to_csv(L1_TABLES / 'per_paper_statistics.csv', index=False)

icc_results = {
    'overall_ICC': ICC_value,
    'sigma2_between': sigma2_between,
    'sigma2_within': sigma2_within,
    'sigma2_total': sigma2_total,
    'F_stat': F_stat,
    'F_p': F_p,
    'n_papers': n_papers,
    'n_total': n_total,
    'feature_ICCs': feature_iccs,
}
checkpoint(icc_results, 'icc_results')

print(f'\n✓ Table saved: per_paper_statistics.csv')

In [ ]:
N_BOOT = 1000
rng = np.random.RandomState(SEED)

boot_iccs = []
for b in range(N_BOOT):
    boot_papers = rng.choice(paper_ids, size=n_papers, replace=True)

    boot_rows = []
    for j, p in enumerate(boot_papers):
        sub = df_cv[df_cv.Paper_ID == p].copy()
        sub['boot_paper'] = f'paper_{j}'  # unique ID for each draw
        boot_rows.append(sub)
    boot_df = pd.concat(boot_rows, ignore_index=True)

    bp_ids = boot_df.boot_paper.unique()
    n_bp = len(bp_ids)
    n_bt = len(boot_df)
    gm = boot_df.log_diameter.mean()

    g_means_b = boot_df.groupby('boot_paper')['log_diameter'].mean()
    g_sizes_b = boot_df.groupby('boot_paper')['log_diameter'].count()

    ss_b = sum(g_sizes_b[p] * (g_means_b[p] - gm)**2 for p in bp_ids)
    ss_w = sum(((boot_df[boot_df.boot_paper == p]['log_diameter'] - g_means_b[p])**2).sum()
               for p in bp_ids)

    df_b = n_bp - 1
    df_w = n_bt - n_bp

    if df_b == 0 or df_w == 0:
        continue

    ms_b = ss_b / df_b
    ms_w = ss_w / df_w
    n0 = (n_bt - sum(g_sizes_b**2) / n_bt) / (n_bp - 1)

    s2_b = max(0, (ms_b - ms_w) / n0)
    s2_w = ms_w
    s2_t = s2_b + s2_w

    if s2_t > 0:
        boot_iccs.append(s2_b / s2_t)

boot_iccs = np.array(boot_iccs)

print('='*70)
print('ICC BOOTSTRAP SENSITIVITY ANALYSIS (1000 resamples)')
print('='*70)
print(f'  Original ICC:      {ICC_value:.4f}')
print(f'  Bootstrap mean:    {boot_iccs.mean():.4f}')
print(f'  Bootstrap median:  {np.median(boot_iccs):.4f}')
print(f'  Bootstrap SD:      {boot_iccs.std():.4f}')
print(f'  95% CI:            [{np.percentile(boot_iccs, 2.5):.4f}, {np.percentile(boot_iccs, 97.5):.4f}]')
print(f'  Min:               {boot_iccs.min():.4f}')
print(f'  Max:               {boot_iccs.max():.4f}')

print(f'\n{"="*70}')
print('CONDITION-LEVEL ICC — Variance within feature subsets')
print(f'{"="*70}')

def compute_icc_subset(df_sub):
    """Compute ICC for a subset of the data."""
    pids = df_sub.Paper_ID.unique()
    if len(pids) < 3:
        return float('nan')

    n_p = len(pids)
    n_t = len(df_sub)
    gm = df_sub.log_diameter.mean()

    g_m = df_sub.groupby('Paper_ID')['log_diameter'].mean()
    g_s = df_sub.groupby('Paper_ID')['log_diameter'].count()

    ss_b = sum(g_s[p] * (g_m[p] - gm)**2 for p in pids)
    ss_w = sum(((df_sub[df_sub.Paper_ID == p]['log_diameter'] - g_m[p])**2).sum()
               for p in pids)

    df_b = n_p - 1
    df_w = n_t - n_p

    if df_b == 0 or df_w == 0:
        return float('nan')

    ms_b = ss_b / df_b
    ms_w = ss_w / df_w
    n0 = (n_t - sum(g_s**2) / n_t) / (n_p - 1)

    s2_b = max(0, (ms_b - ms_w) / n0)
    s2_w = ms_w
    s2_t = s2_b + s2_w

    return s2_b / s2_t if s2_t > 0 else 0

conc_bins = [(0, 5, 'Low (0-5%)'), (5, 10, 'Mid (5-10%)'),
             (10, 20, 'High (10-20%)'), (20, 35, 'Very high (20-35%)')]

print(f'\nBy concentration range:')
print(f'  {"Range":20s} | {"n":>4s} | {"Papers":>6s} | {"ICC":>6s}')
print(f'  {"-"*45}')
cond_iccs = {}
for lo, hi, label in conc_bins:
    sub = df_cv[(df_cv.Polymer_conc_wt >= lo) & (df_cv.Polymer_conc_wt < hi)]
    if len(sub) > 0:
        icc_sub = compute_icc_subset(sub)
        cond_iccs[f'conc_{label}'] = icc_sub
        print(f'  {label:20s} | {len(sub):4d} | {sub.Paper_ID.nunique():6d} | {icc_sub:6.4f}')

pres_bins = [(0, 1.5, 'Low (0-1.5 bar)'), (1.5, 3, 'Mid (1.5-3 bar)'),
             (3, 5, 'High (3-5 bar)'), (5, 7, 'Very high (5-7 bar)')]

print(f'\nBy pressure range:')
print(f'  {"Range":20s} | {"n":>4s} | {"Papers":>6s} | {"ICC":>6s}')
print(f'  {"-"*45}')
for lo, hi, label in pres_bins:
    sub = df_cv[(df_cv.Air_pressure_bar_std >= lo) & (df_cv.Air_pressure_bar_std < hi)]
    if len(sub) > 0:
        icc_sub = compute_icc_subset(sub)
        cond_iccs[f'pres_{label}'] = icc_sub
        print(f'  {label:20s} | {len(sub):4d} | {sub.Paper_ID.nunique():6d} | {icc_sub:6.4f}')

print(f'\nBy polymer (top 5):')
print(f'  {"Polymer":20s} | {"n":>4s} | {"Papers":>6s} | {"ICC":>6s}')
print(f'  {"-"*45}')
top_polys = df_cv.Polymer_type.value_counts().head(5).index
for poly in top_polys:
    sub = df_cv[df_cv.Polymer_type == poly]
    if sub.Paper_ID.nunique() >= 3:
        icc_sub = compute_icc_subset(sub)
        cond_iccs[f'poly_{poly}'] = icc_sub
        print(f'  {poly:20s} | {len(sub):4d} | {sub.Paper_ID.nunique():6d} | {icc_sub:6.4f}')
    else:
        print(f'  {poly:20s} | {len(sub):4d} | {sub.Paper_ID.nunique():6d} | (too few papers)')

pie_data = {
    'between_paper_pct': sigma2_between / sigma2_total * 100,
    'within_paper_pct': sigma2_within / sigma2_total * 100,
}

print(f'\n{"="*60}')
print('VARIANCE DECOMPOSITION (for pie chart)')
print(f'{"="*60}')
print(f'  Between-paper: {pie_data["between_paper_pct"]:.1f}%')
print(f'  Within-paper:  {pie_data["within_paper_pct"]:.1f}%')

icc_sensitivity = {
    'bootstrap_iccs': boot_iccs.tolist(),
    'bootstrap_mean': float(boot_iccs.mean()),
    'bootstrap_median': float(np.median(boot_iccs)),
    'bootstrap_std': float(boot_iccs.std()),
    'ci_95': [float(np.percentile(boot_iccs, 2.5)), float(np.percentile(boot_iccs, 97.5))],
    'condition_iccs': cond_iccs,
    'pie_data': pie_data,
}
checkpoint(icc_sensitivity, 'icc_sensitivity')

In [ ]:
import statsmodels.api as sm
from statsmodels.regression.mixed_linear_model import MixedLM

formula = 'log_diameter ~ ' + ' + '.join(ML_FEATURES)
md = MixedLM.from_formula(formula, groups='Paper_ID', data=df_cv)
mdf = md.fit(reml=True)

sigma2_paper = float(mdf.cov_re.iloc[0, 0])
sigma2_resid = float(mdf.scale)
conditional_ICC = sigma2_paper / (sigma2_paper + sigma2_resid)

print(f'sigma2_paper (random intercept) = {sigma2_paper:.4f}')
print(f'sigma2_resid                    = {sigma2_resid:.4f}')
print(f'Conditional ICC                 = {conditional_ICC:.3f}')
print(f'{conditional_ICC*100:.1f}% of residual variance is between-paper after adjusting for features')

y = df_cv['log_diameter'].values
soln_feats = ['Polymer_conc_wt', 'intrinsic_viscosity_dLg', 'c_over_cstar']
solv_feats = soln_feats + ['Solvent_ST_mNm', 'Solvent_BP_C']
proc_feats = ML_FEATURES

r2_solution         = sm.OLS(y, sm.add_constant(df_cv[soln_feats].values)).fit().rsquared
r2_solution_solvent = sm.OLS(y, sm.add_constant(df_cv[solv_feats].values)).fit().rsquared
r2_all              = sm.OLS(y, sm.add_constant(df_cv[proc_feats].values)).fit().rsquared

print(f'OLS R2 solution            = {r2_solution:.4f}')
print(f'OLS R2 solution + solvent  = {r2_solution_solvent:.4f} (+{(r2_solution_solvent-r2_solution)*100:.1f}%)')
print(f'OLS R2 all 8 features      = {r2_all:.4f} (+{(r2_all-r2_solution_solvent)*100:.1f}%)')
print(f'Unexplained                = {(1-r2_all)*100:.1f}%')

icc_extended = {
    'conditional_ICC': conditional_ICC,
    'sigma2_paper': sigma2_paper,
    'sigma2_resid': sigma2_resid,
    'OLS_R2_solution': r2_solution,
    'OLS_R2_solution_solvent': r2_solution_solvent,
    'OLS_R2_all': r2_all,
}
checkpoint(icc_extended, 'icc_extended_results')
(ROOT / '1_Level1_CV' / 'tables').mkdir(parents=True, exist_ok=True)
pd.DataFrame([icc_extended]).to_csv(ROOT / '1_Level1_CV' / 'tables' / 'T_icc_extended.csv', index=False)

In [ ]:
import shap

SHAP_MODELS = ['ExtraTrees', 'RandomForest', 'XGBoost', 'LightGBM', 'CatBoost', 'SVR']

shap_values_all = load_checkpoint('shap_values_all') or {}

print('='*70)
print('SHAP ANALYSIS — 6 MODELS')
print('='*70)

for name in SHAP_MODELS:
    if name in shap_values_all:
        print(f'⏭  {name}: loaded from checkpoint')
        continue

    t0 = time.time()
    model = final_models_ml[name]

    if name == 'SVR':
        print(f'▶ {name} (KernelSHAP, 100 background samples)...')
        np.random.seed(SEED)
        bg_idx = np.random.choice(len(X_cv_scaled), 100, replace=False)
        background = X_cv_scaled[bg_idx]
        explainer = shap.KernelExplainer(model.predict, background)
        sv = explainer.shap_values(X_cv_scaled, nsamples=200)
    else:
        print(f'▶ {name} (TreeSHAP)...')
        explainer = shap.TreeExplainer(model)
        sv = explainer.shap_values(X_cv_scaled)

    elapsed = time.time() - t0

    shap_values_all[name] = {
        'shap_values': sv,              # (341, 8) array
        'expected_value': float(explainer.expected_value) if np.isscalar(explainer.expected_value) else float(explainer.expected_value[0]),
        'feature_names': FEATURES,
    }

    mean_abs = np.abs(sv).mean(axis=0)
    total = mean_abs.sum()
    print(f'  ✓ {name} ({elapsed:.1f}s)')
    print(f'    Top 3: ', end='')
    rank = np.argsort(-mean_abs)
    for r in rank[:3]:
        print(f'{FEATURES[r]} ({mean_abs[r]/total*100:.1f}%), ', end='')
    print()

    checkpoint(shap_values_all, 'shap_values_all')

print(f'\n▶ ElasticNet coefficients (linear model — no SHAP needed):')
en_model = final_models_ml['ElasticNet']
en_coefs = en_model.coef_
total_abs = np.abs(en_coefs).sum()
for i, f in enumerate(FEATURES):
    pct = np.abs(en_coefs[i]) / total_abs * 100
    print(f'    {f:28s}: coef={en_coefs[i]:+.4f} ({pct:.1f}%)')

shap_values_all['ElasticNet_coefs'] = {
    'coefficients': en_coefs.tolist(),
    'intercept': float(en_model.intercept_),
    'feature_names': FEATURES,
}
checkpoint(shap_values_all, 'shap_values_all')

print(f'\n{"="*70}')
print('MEAN |SHAP| FEATURE IMPORTANCE (% of total)')
print(f'{"="*70}')

header = f'{"Feature":28s} |'
for name in SHAP_MODELS:
    header += f' {name:>10s} |'
print(header)
print('-' * len(header))

importance_matrix = {}
for name in SHAP_MODELS:
    sv = shap_values_all[name]['shap_values']
    mean_abs = np.abs(sv).mean(axis=0)
    total = mean_abs.sum()
    importance_matrix[name] = (mean_abs / total * 100).tolist()

for i, f in enumerate(FEATURES):
    row = f'{f:28s} |'
    for name in SHAP_MODELS:
        row += f' {importance_matrix[name][i]:9.1f}% |'
    print(row)

imp_rows = []
for i, f in enumerate(FEATURES):
    r = {'Feature': f}
    for name in SHAP_MODELS:
        r[name] = round(importance_matrix[name][i], 1)
    imp_rows.append(r)
pd.DataFrame(imp_rows).to_csv(L1_TABLES / 'shap_feature_importance.csv', index=False)

print(f'\n✓ Table saved: shap_feature_importance.csv')

In [ ]:
best_shap_model = 'ExtraTrees'
sv = shap_values_all[best_shap_model]['shap_values']
ev = shap_values_all[best_shap_model]['expected_value']

oof_data = load_checkpoint('oof_predictions_all')
oof_pred = oof_data[best_shap_model]['y_pred']
oof_true = oof_data[best_shap_model]['y_true']

errors_nm = np.abs(np.exp(oof_true) - np.exp(oof_pred))
pct_errors = np.abs(np.exp(oof_true) - np.exp(oof_pred)) / np.exp(oof_true) * 100

idx_best = np.argmin(pct_errors)

idx_worst = np.argmax(pct_errors)

median_err = np.median(pct_errors)
idx_median = np.argmin(np.abs(pct_errors - median_err))

waterfall_samples = {
    'best': {
        'index': int(idx_best),
        'actual_nm': float(np.exp(oof_true[idx_best])),
        'predicted_nm': float(np.exp(oof_pred[idx_best])),
        'error_pct': float(pct_errors[idx_best]),
        'shap_values': sv[idx_best].tolist(),
        'feature_values': X_cv_scaled[idx_best].tolist(),
        'raw_feature_values': df_cv[FEATURES].iloc[idx_best].tolist(),
        'paper': df_cv.iloc[idx_best].Paper_ID,
        'polymer': df_cv.iloc[idx_best].Polymer_type,
    },
    'median': {
        'index': int(idx_median),
        'actual_nm': float(np.exp(oof_true[idx_median])),
        'predicted_nm': float(np.exp(oof_pred[idx_median])),
        'error_pct': float(pct_errors[idx_median]),
        'shap_values': sv[idx_median].tolist(),
        'feature_values': X_cv_scaled[idx_median].tolist(),
        'raw_feature_values': df_cv[FEATURES].iloc[idx_median].tolist(),
        'paper': df_cv.iloc[idx_median].Paper_ID,
        'polymer': df_cv.iloc[idx_median].Polymer_type,
    },
    'worst': {
        'index': int(idx_worst),
        'actual_nm': float(np.exp(oof_true[idx_worst])),
        'predicted_nm': float(np.exp(oof_pred[idx_worst])),
        'error_pct': float(pct_errors[idx_worst]),
        'shap_values': sv[idx_worst].tolist(),
        'feature_values': X_cv_scaled[idx_worst].tolist(),
        'raw_feature_values': df_cv[FEATURES].iloc[idx_worst].tolist(),
        'paper': df_cv.iloc[idx_worst].Paper_ID,
        'polymer': df_cv.iloc[idx_worst].Polymer_type,
    },
}

print('='*70)
print(f'WATERFALL EXAMPLES — {best_shap_model}')
print('='*70)

for label in ['best', 'median', 'worst']:
    ws = waterfall_samples[label]
    print(f'\n  {label.upper()} predicted sample (idx={ws["index"]}):')
    print(f'    Paper: {ws["paper"]}, Polymer: {ws["polymer"]}')
    print(f'    Actual: {ws["actual_nm"]:.0f} nm, Predicted: {ws["predicted_nm"]:.0f} nm, '
          f'Error: {ws["error_pct"]:.1f}%')
    print(f'    SHAP contributions:')
    sv_sample = np.array(ws['shap_values'])
    rank = np.argsort(-np.abs(sv_sample))
    for r in rank:
        raw_val = ws['raw_feature_values'][r]
        print(f'      {FEATURES[r]:28s}: SHAP={sv_sample[r]:+.4f} (value={raw_val:.3f})')
    print(f'    Base value: {ev:.4f}, Sum SHAP: {sv_sample.sum():+.4f}, '
          f'Prediction: {ev + sv_sample.sum():.4f}')

checkpoint(waterfall_samples, 'waterfall_samples')

print(f'\n{"="*70}')
print('CONSENSUS SHAP RANKING (mean across 5 tree models)')
print(f'{"="*70}')

tree_models = ['ExtraTrees', 'RandomForest', 'XGBoost', 'LightGBM', 'CatBoost']
consensus = np.zeros(len(FEATURES))
for name in tree_models:
    sv_m = shap_values_all[name]['shap_values']
    mean_abs = np.abs(sv_m).mean(axis=0)
    consensus += mean_abs / mean_abs.sum()
consensus /= len(tree_models)
consensus_pct = consensus / consensus.sum() * 100

rank = np.argsort(-consensus_pct)
for i, r in enumerate(rank):
    print(f'  {i+1}. {FEATURES[r]:28s}: {consensus_pct[r]:.1f}%')

In [ ]:
from sklearn.model_selection import KFold, cross_val_score
from sklearn.ensemble import ExtraTreesRegressor

kf_reg = KFold(n_splits=5, shuffle=True, random_state=SEED)
best_et_params = best_params_ml['ExtraTrees'].copy()
best_et_params['random_state'] = SEED

model_full = ExtraTreesRegressor(**best_et_params)
scores_full = cross_val_score(model_full, X_cv_scaled, y_cv, cv=kf_reg, scoring='r2')
r2_full = scores_full.mean()

print('='*60)
print('FEATURE ABLATION — ExtraTrees')
print('='*60)
print(f'Full model R²: {r2_full:.4f} ± {scores_full.std():.4f}')
print(f'\n{"Feature":28s} | {"R² w/o":>8s} | {"Δ R²":>8s} | {"Impact":>8s}')
print('-'*60)

ablation_results = {}
for i, feat in enumerate(FEATURES):
    mask = [j for j in range(len(FEATURES)) if j != i]
    X_abl = X_cv_scaled[:, mask]

    model_abl = ExtraTreesRegressor(**best_et_params)
    scores_abl = cross_val_score(model_abl, X_abl, y_cv, cv=kf_reg, scoring='r2')
    r2_abl = scores_abl.mean()
    delta = r2_full - r2_abl

    impact = 'critical' if delta > 0.02 else 'moderate' if delta > 0.005 else 'marginal' if delta > 0 else 'redundant'

    ablation_results[feat] = {
        'r2_without': round(r2_abl, 4),
        'r2_drop': round(delta, 4),
        'r2_std': round(scores_abl.std(), 4),
    }
    print(f'{feat:28s} | {r2_abl:8.4f} | {delta:+8.4f} | {impact}')

checkpoint(ablation_results, 'ablation_results')
pd.DataFrame([
    {'Feature': f, **v} for f, v in ablation_results.items()
]).to_csv(L1_TABLES / 'feature_ablation.csv', index=False)

print(f'\n✓ Table saved: feature_ablation.csv')